# Ergänzende Modellvergleiche

Dieses Notebook erhält den Analyseablauf vor der Integration von NMF mit fünfzehn Topics. Es enthält die Vergleiche von Textnormalisierung, Vokabularfilter, Anzahl der Topics, Count und TF-IDF, Initialisierung und LDA-Trainingsumfang.

Die damaligen Einstellungen bleiben erhalten. Die Ergebnisübersicht am Ende gehört zur Referenz mit zwölf NMF-Topics. Die Auswahl der neuen Variante und die wesentlichen Befunde sind in der README dokumentiert.

Die aktuelle Analyse steht in [01_themenanalyse.ipynb](01_themenanalyse.ipynb). Dieses Notebook wird dafür nicht benötigt. Die Zellen hier sind in einer eigenen Sitzung von oben nach unten auszuführen. Ausgaben werden unter `outputs/ergaenzende_vergleiche/` gespeichert.

Noch frühere Versuche sind in [02_bisherige_vergleiche.ipynb](02_bisherige_vergleiche.ipynb) dokumentiert.

Die ursprünglichen Vergleiche wurden mit der Originaldatei durchgeführt. Bei einer erneuten Ausführung wird die mitgelieferte bereinigte CSV verwendet. Einzelne Ergebnisse können deshalb von den dokumentierten Werten abweichen. Die damaligen Modellparameter und Textregeln bleiben erhalten.


In [ ]:
# Schritt 01 Umgebung und Arbeitsordner prüfen

from pathlib import Path
import sys
import re

import pandas as pd
import numpy as np
import sklearn
from IPython.display import display

print("Python executable:")
print(sys.executable)

print("\nPandas Version:")
print(pd.__version__)

print("\nNumPy Version:")
print(np.__version__)

print("\nScikit Learn Version:")
print(sklearn.__version__)

print("\nAktueller Arbeitsordner:")
print(Path.cwd())


In [ ]:
# Schritt 02 Konfiguration

CSV_NAME = "maengelmelder_konstanz_final_9_bereinigt.csv"
TEXT_COLUMN = "text_raw"
ID_COLUMN = "id"

# Kurze Meldungen bleiben erhalten, sofern nach dem Preprocessing ein Token verbleibt.
MIN_WORDS_RAW = 1
MIN_TOKENS_CLEAN = 1
MIN_TOKEN_LENGTH = 2

# Parameter der Vektorisierung
MIN_DF = 5
MAX_DF = 0.90
NGRAM_RANGE = (1, 1)
TOKEN_PATTERN = r"(?u)\b[a-zäöüß][a-zäöüß]+\b"
SUBLINEAR_TF = True

# Referenzmodelle vor dem Vergleich
N_TOPICS = 10
N_TOPICS = 10
RANDOM_STATE = 42
LDA_MAX_ITER = 20
NMF_MAX_ITER = 500

# Modelle für die Ergebnisübersichten
SELECTED_TOPIC_COUNTS = {"LDA": 10, "NMF": 12}

# Die LDA-Priors bleiben für alle verglichenen Anzahlen von Topics gleich.
LDA_DOC_TOPIC_PRIOR = 0.1
LDA_TOPIC_WORD_PRIOR = 0.1

# Erst nach der inhaltlichen Prüfung der Bezeichnungen für das aktuelle Modell aktivieren
TOPIC_LABELS_REVIEWED = True

# Deutsche Stopwörter aus dem bisherigen Prototyp
GERMAN_STOPWORDS = {
    "aber", "alle", "allem", "allen", "aller", "alles", "als", "also", "am", "an",
    "andere", "anderem", "anderen", "anderer", "anderes", "auch", "auf", "aus",
    "bei", "bin", "bis", "bist", "da", "damit", "dann", "der", "den", "des",
    "dem", "die", "das", "dass", "daß", "dazu", "dein", "deine", "deinem",
    "deinen", "deiner", "deines", "denn", "dich", "dir", "du", "dies", "diese",
    "diesem", "diesen", "dieser", "dieses", "doch", "dort", "durch", "ein",
    "eine", "einem", "einen", "einer", "eines", "einige", "einigen", "einiger",
    "einiges", "einmal", "er", "ihn", "ihm", "es", "etwas", "euer", "eure",
    "eurem", "euren", "eurer", "eures", "für", "gegen", "gewesen", "hab",
    "habe", "haben", "hat", "hatte", "hatten", "hier", "hin", "hinter", "ich",
    "mich", "mir", "ihr", "ihre", "ihrem", "ihren", "ihrer", "ihres", "euch",
    "im", "in", "indem", "ins", "ist", "jede", "jedem", "jeden", "jeder",
    "jedes", "jene", "jenem", "jenen", "jener", "jenes", "jetzt", "kann",
    "kein", "keine", "keinem", "keinen", "keiner", "keines", "können", "könnte",
    "machen", "man", "manche", "manchem", "manchen", "mancher", "manches",
    "mein", "meine", "meinem", "meinen", "meiner", "meines", "mit", "muss",
    "musste", "nach", "nicht", "nichts", "noch", "nun", "nur", "ob", "oder",
    "ohne", "sehr", "sein", "seine", "seinem", "seinen", "seiner", "seines",
    "selbst", "sich", "sie", "ihnen", "sind", "so", "solche", "solchem",
    "solchen", "solcher", "solches", "soll", "sollte", "sondern", "sonst",
    "über", "um", "und", "uns", "unser", "unsere", "unserem", "unseren",
    "unserer", "unseres", "unter", "viel", "vom", "von", "vor", "während",
    "war", "waren", "warst", "was", "weg", "weil", "weiter", "welche",
    "welchem", "welchen", "welcher", "welches", "wenn", "werde", "werden",
    "wie", "wieder", "will", "wir", "wird", "wirst", "wo", "wollen", "wollte",
    "würde", "würden", "zu", "zum", "zur", "zwar", "zwischen"
}

# Zusammengeführte Ergänzungen aus den bisherigen Versuchen.
DOMAIN_STOPWORDS = {
    "abends", "abgeholt", "abgelaufen", "abgelegt", "abgeschlossen", "abgestellt",
    "abgestellte", "abgestellter", "abgestelltes", "abholung", "adresssuche", "aktuell",
    "allerdings", "allmannsdorf", "allmannshoehe", "alte", "alter", "altes", "andere",
    "anderen", "ausgefallen", "autom", "beethovenstr", "befinden", "befindet", "beiden", "beim",
    "bereich", "bereits", "beschaedigt", "beschädigt", "bild", "bilder", "bitte", "brennen",
    "brennt", "ca", "circa", "dank", "danke", "dankeschoen", "dankeschön", "defekt", "defekte",
    "defekten", "defekter", "dienstag", "direkt", "donnerstag", "dorffriedhofsweg", "dunkel",
    "ecke", "entfernen", "entsorgt", "ermittelt", "etc", "flurweg", "foto", "fotos", "freitag",
    "frieda", "fuer", "funktion", "funktioniert", "fuss", "ganze", "gegenueber", "gegenüber",
    "gehen", "geht", "gemeldet", "gestern", "guten", "haengt", "hallo", "hans",
    "hansegartenstrasse", "haus", "hausnummer", "heugaessle", "heute", "hinten", "hoehe",
    "hoehenweg", "hängt", "höhe", "immer", "jahr", "jahren", "kaputt", "kaputte", "kaputtes",
    "kaum", "koennen", "koennte", "kommen", "kommt", "komplett", "konstanz", "konstanzer",
    "kreuzlinger", "können", "könnte", "laengerem", "leider", "leuchte", "leuchten", "leuchtet",
    "liegen", "liegt", "litzelstetten", "macairestrasse", "mainaustrasse", "mal", "mannheimer",
    "max", "mehr", "mehrere", "mehreren", "mehrfach", "mfg", "mindestens", "mittags",
    "mittwoch", "moeglich", "moegliche", "moeglicherweise", "monat", "monaten", "montag",
    "morgen", "möglich", "nacht", "naehe", "nahe", "neben", "nr", "nummer", "oben", "ort",
    "peter", "richtung", "samstag", "schaden", "schadhaft", "schaeden", "schneckenburgstrasse",
    "schoen", "schon", "schwach", "schön", "seit", "seite", "siehe", "sieht", "sigrist",
    "sonnentauweg", "sonntag", "stadt", "stark", "stehen", "steht", "steinstrasse", "stelle",
    "str", "stromeyer", "tag", "tage", "tagen", "teil", "teile", "thumb", "ueber", "unten",
    "untere", "unteren", "urisberg", "vermutlich", "viele", "vielen", "vorne", "waere",
    "waeren", "weiteres", "woche", "wochen", "worden", "wuerde", "wuerden", "wurde", "wurden",
    "wäre", "wären", "zeit", "zwei",
}

STOPWORDS_FINAL = GERMAN_STOPWORDS | DOMAIN_STOPWORDS

# Bisherige Zuordnung ausgewählter Schreibweisen und Begriffe
CANONICAL_TERMS_FINAL = {
    "abfaelle": "müll",
    "abfall": "müll",
    "abfälle": "müll",
    "aeste": "äste",
    "baeume": "baum",
    "beschaedigte": "beschädigt",
    "beschaedigten": "beschädigt",
    "beschädigte": "beschädigt",
    "bäume": "baum",
    "container": "müllcontainer",
    "fahrrads": "fahrrad",
    "fahrraeder": "fahrrad",
    "fahrräder": "fahrrad",
    "fuer": "für",
    "fussgaenger": "fußgänger",
    "fussgaengerampel": "ampel",
    "fussweg": "gehweg",
    "fusswege": "gehweg",
    "fußgängerampel": "ampel",
    "fußweg": "gehweg",
    "fußwege": "gehweg",
    "gefaehrlich": "gefährlich",
    "gehwege": "gehweg",
    "grobmuell": "grobmüll",
    "gruen": "grün",
    "gullideckel": "gulli",
    "gullis": "gulli",
    "haengt": "hängt",
    "hecken": "hecke",
    "heckenwuchs": "hecke",
    "koennen": "können",
    "laeuft": "läuft",
    "lampe": "beleuchtung",
    "lampen": "beleuchtung",
    "laterne": "beleuchtung",
    "laternen": "beleuchtung",
    "leuchte": "beleuchtung",
    "moeglich": "möglich",
    "muell": "müll",
    "muelleimer": "mülleimer",
    "müllcontainer": "müll",
    "mülltonne": "müll",
    "mülltonnen": "müll",
    "radwege": "radweg",
    "schilder": "schild",
    "straeucher": "sträucher",
    "strasse": "straße",
    "strassen": "straße",
    "strassenbeleuchtung": "beleuchtung",
    "strassenlampe": "beleuchtung",
    "strassenlampen": "beleuchtung",
    "strassenlaterne": "beleuchtung",
    "strassenlaternen": "beleuchtung",
    "strassenleuchte": "beleuchtung",
    "strassenleuchten": "beleuchtung",
    "strassenseiten": "straßenseiten",
    "strassensteiten": "straßenseiten",
    "strauch": "sträucher",
    "straßenbeleuchtung": "beleuchtung",
    "straßenlaterne": "beleuchtung",
    "straßenlaternen": "beleuchtung",
    "ueber": "über",
    "ueberfuellt": "überfüllt",
    "umgestuerzt": "umgestürzt",
    "verkehrsschild": "schild",
    "verkehrsschilder": "schild",
    "verschmutzte": "verschmutzt",
    "verschmutzten": "verschmutzt",
    "verunreinigt": "verschmutzt",
    "waechst": "wächst",
    "waere": "wäre",
    "zerstoert": "zerstört",
}


In [ ]:
# Schritt 03 CSV-Datei laden

from pathlib import Path
import pandas as pd

possible_paths = [
    Path("data") / "raw" / CSV_NAME,
    Path("..") / "data" / "raw" / CSV_NAME,
]

csv_path = None

for path in possible_paths:
    print("Prüfe:", path.resolve())
    if path.exists():
        csv_path = path
        break

if csv_path is None:
    raise FileNotFoundError(
        "CSV wurde nicht gefunden. Prüfe, ob die Datei im Ordner data/raw liegt."
    )

print("\nGefundene CSV:")
print(csv_path.resolve())

# Der Projektordner ergibt sich aus dem Pfad data/raw der gefundenen CSV.
PROJECT_DIR = csv_path.parent.resolve().parent.parent

df = pd.read_csv(csv_path)

print("\nShape:")
print(df.shape)

print("\nSpalten:")
print(list(df.columns))

df.head(3)


In [ ]:
# Schritt 04 Data Profiling

required_columns = [
    "id",
    "date_created",
    TEXT_COLUMN,
    "category_parent",
    "category_child",
    "responsible",
    "lat",
    "lon",
    "address_input",
    "detail_url",
]

missing_required = [col for col in required_columns if col not in df.columns]

if missing_required:
    raise ValueError(f"Folgende erwartete Spalten fehlen: {missing_required}")

print("Anzahl Zeilen:", len(df))

print("\nFehlende Werte je Spalte:")
print(df.isna().sum())

print("\nFehlende Werte in text_raw:")
print(df[TEXT_COLUMN].isna().sum())

print("\nDubletten in text_raw:")
print(df[TEXT_COLUMN].duplicated().sum())

print("\nKategorien in category_parent:")
print(df["category_parent"].value_counts(dropna=False))

print("\nKategorien in category_child:")
print(df["category_child"].value_counts(dropna=False).head(25))

print("\nTextlängen in Wörtern:")
word_counts_raw = df[TEXT_COLUMN].dropna().astype(str).str.split().str.len()
print(word_counts_raw.describe())

print("\nBeispiele aus text_raw:")
for i, text in enumerate(df[TEXT_COLUMN].dropna().astype(str).head(5), start=1):
    print(f"\nBeispiel {i}:")
    print(text[:500])


In [ ]:
# Schritt 05 Arbeitsdatensatz erstellen

def prepare_records(
    source_df: pd.DataFrame,
    text_column: str,
    min_words: int,
    id_column: str,
):
    """Doppelte Datensätze entfernen und Meldungen mit unterschiedlichen IDs beibehalten."""
    if not source_df.columns.is_unique:
        raise ValueError("Die Spaltennamen müssen eindeutig sein.")
    missing_columns = [col for col in (text_column, id_column) if col not in source_df.columns]
    if missing_columns:
        raise ValueError(f"Erforderliche Spalten fehlen: {missing_columns}")
    if text_column == id_column:
        raise ValueError("Textspalte und ID-Spalte müssen unterschiedlich sein.")
    if not isinstance(min_words, int) or isinstance(min_words, bool) or min_words < 1:
        raise ValueError("min_words muss eine ganze Zahl größer oder gleich 1 sein.")

    counts = {"input": len(source_df)}

    # Fehlende IDs nicht als gemeinsame Meldung behandeln.
    id_keys = source_df[id_column].astype("string").str.strip()
    if id_keys.isna().any() or id_keys.eq("").any():
        raise ValueError("Meldungs-IDs fehlen. Die betroffenen Datensätze müssen geprüft werden.")

    # Nur über alle Spalten identische Datensätze entfernen, einschließlich der ID.
    work = source_df.drop_duplicates(keep="first").copy()
    counts["duplicates"] = len(source_df) - len(work)

    # Bei unterschiedlichen Inhalten zur selben ID keinen Datensatz willkürlich auswählen.
    if work[id_column].astype("string").str.strip().duplicated(keep=False).any():
        raise ValueError(
            "Zu einer Meldungs-ID liegen unterschiedliche Datensätze vor. "
            "Die Unterschiede müssen vor der Analyse geprüft werden."
        )

    before = len(work)
    work = work.dropna(subset=[text_column]).copy()
    counts["missing"] = before - len(work)
    work[text_column] = work[text_column].astype(str).str.strip()

    before = len(work)
    work = work.loc[work[text_column].str.len() > 0].copy()
    counts["empty"] = before - len(work)

    # Gleiche Texte mit unterschiedlichen IDs zählen weiterhin als einzelne Meldungen.
    work["word_count_raw"] = work[text_column].str.split().str.len()
    before = len(work)
    work = work.loc[work["word_count_raw"] >= min_words].copy()
    counts["raw_short"] = before - len(work)
    counts["remaining"] = len(work)
    counts["text_repetitions"] = int(work[text_column].duplicated().sum())

    if counts["input"] != sum(counts[key] for key in (
        "duplicates", "missing", "empty", "raw_short", "remaining"
    )):
        raise RuntimeError("Die dokumentierten Zeilenzahlen sind nicht konsistent.")

    return work.reset_index(drop=True), counts


df_work, record_counts = prepare_records(df, TEXT_COLUMN, MIN_WORDS_RAW, ID_COLUMN)

print("Ausgangszeilen:", record_counts["input"])
print("Entfernte vollständig identische Datensätze:", record_counts["duplicates"])
print("Entfernte fehlende Texte:", record_counts["missing"])
print("Entfernte leere Texte:", record_counts["empty"])
print("Entfernte Texte unter der Mindestlänge:", record_counts["raw_short"])
print("Meldungen im Arbeitsdatensatz:", record_counts["remaining"])
print("Weitere Meldungen mit gleichem Text, beibehalten:", record_counts["text_repetitions"])

# Unterschiedliche IDs mit gleichem Text bleiben erhalten, eine identische Zeile entfällt.
_test_data = pd.DataFrame({
    ID_COLUMN: [1, 2, 3, 4, 5, 6, 7, 8, 8],
    TEXT_COLUMN: [None, np.nan, pd.NA, "", "  ", " Müll ", "Müll", "Lampe defekt", "Lampe defekt"],
})
_test_work, _test_counts = prepare_records(_test_data, TEXT_COLUMN, 1, ID_COLUMN)
assert _test_work[ID_COLUMN].tolist() == [6, 7, 8]
assert _test_work[TEXT_COLUMN].tolist() == ["Müll", "Müll", "Lampe defekt"]
assert _test_counts == {
    "input": 9, "duplicates": 1, "missing": 3, "empty": 2,
    "raw_short": 0, "remaining": 3, "text_repetitions": 1,
}
print("Funktionstest für Fehlwerte und Dubletten: bestanden")

df_work[[ID_COLUMN, TEXT_COLUMN, "category_parent", "category_child", "word_count_raw"]].head(10)

In [ ]:
# Schritt 06 Ausgabeordner festlegen

OUTPUT_DIR = PROJECT_DIR / "outputs" / "ergaenzende_vergleiche"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output-Ordner:")
print(OUTPUT_DIR)


In [ ]:
# Schritt 07 Text Preprocessing definieren

# Die Endungsregeln erkennen mögliche Ortsangaben, aber nicht alle Ortsnamen.
def is_location_token(token: str) -> bool:
    # Mögliche Straßennamen anhand der Endung erkennen und "straße" ausnehmen.
    if token == "straße":
        return False

    if token.endswith(("strasse", "straße", "str")) and len(token) > 5:
        return True

    # Mögliche Wegnamen erkennen und die folgenden allgemeinen Wegbegriffe ausnehmen.
    if token in {"gehweg", "radweg", "fahrradweg"}:
        return False

    if token.endswith("weg") and len(token) > 5:
        return True

    return False

def clean_text(text: str) -> str:
    text = str(text).lower()

    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    text = re.sub(r"[^a-zäöüß\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    tokens = []

    for token in text.split():
        token = CANONICAL_TERMS_FINAL.get(token, token)

        if is_location_token(token):
            continue

        if token not in STOPWORDS_FINAL and len(token) >= MIN_TOKEN_LENGTH:
            tokens.append(token)

    return " ".join(tokens)


In [ ]:
# Schritt 08 Text Preprocessing anwenden

# Für jeden Lauf eine neue Kopie des Arbeitsdatensatzes erzeugen.
df_model = df_work.copy()
df_model["text_clean"] = df_model[TEXT_COLUMN].apply(clean_text)
df_model["word_count_clean"] = df_model["text_clean"].str.split().str.len()

before_clean_filter = len(df_model)
df_model = df_model[df_model["word_count_clean"] >= MIN_TOKENS_CLEAN].copy()
after_clean_filter = len(df_model)
df_model = df_model.reset_index(drop=True)

print("Zeilen vor Textfilter:", before_clean_filter)
print("Zeilen nach Textfilter:", after_clean_filter)
print("Entfernte Meldungen:", before_clean_filter - after_clean_filter)
print("\nVerteilung der Wortanzahl nach Text Preprocessing:")
print(df_model["word_count_clean"].describe())


In [ ]:
# Schritt 09 Bereinigte Texte anhand von Beispielen ansehen

print("Beispiele vorher/nachher:")

for _, row in df_model[[TEXT_COLUMN, "text_clean"]].head(10).iterrows():
    print("\nRAW:")
    print(row[TEXT_COLUMN][:300])
    print("CLEAN:")
    print(row["text_clean"])


In [ ]:
# Schritt 10 Texte mit CountVectorizer und TfidfVectorizer vektorisieren

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

texts_clean = df_model["text_clean"].tolist()

count_vectorizer = CountVectorizer(
    min_df=MIN_DF,
    max_df=MAX_DF,
    token_pattern=TOKEN_PATTERN,
    ngram_range=NGRAM_RANGE
)

tfidf_vectorizer = TfidfVectorizer(
    min_df=MIN_DF,
    max_df=MAX_DF,
    token_pattern=TOKEN_PATTERN,
    ngram_range=NGRAM_RANGE,
    sublinear_tf=SUBLINEAR_TF
)

X_count = count_vectorizer.fit_transform(texts_clean)
X_tfidf = tfidf_vectorizer.fit_transform(texts_clean)

print("Anzahl Dokumente:", len(texts_clean))
print("Count-Matrix:", X_count.shape)
print("TF-IDF-Matrix:", X_tfidf.shape)

print("\nBeispiele aus Count-Vokabular:")
print(count_vectorizer.get_feature_names_out()[:30])

print("\nBeispiele aus TF-IDF-Vokabular:")
print(tfidf_vectorizer.get_feature_names_out()[:30])


In [ ]:
# Kontrolle vor der Modellierung

print("Eingelesene Meldungen:", len(df))
print("Meldungen nach Text Preprocessing:", len(df_model))
print("Count-Matrix:", X_count.shape)
print("TF-IDF-Matrix:", X_tfidf.shape)
print(
    "Meldungen ohne Merkmale in der Count-Matrix:",
    int((X_count.getnnz(axis=1) == 0).sum())
)
print(
    "Meldungen ohne Merkmale in der TF-IDF-Matrix:",
    int((X_tfidf.getnnz(axis=1) == 0).sum())
)

# Kontrollieren, dass nach dem Preprocessing keine leeren Texte verbleiben.
assert df_model[TEXT_COLUMN].notna().all()
assert (df_model["word_count_clean"] >= MIN_TOKENS_CLEAN).all()
assert len(df_model) == X_count.shape[0] == X_tfidf.shape[0]

# Kurze, inhaltliche Testmeldungen bleiben mit der neuen Mindestlänge erhalten.
assert len(clean_text("Lampe defekt").split()) >= MIN_TOKENS_CLEAN
assert len(clean_text("Müll").split()) >= MIN_TOKENS_CLEAN
assert clean_text("Bitte danke") == ""
assert clean_text("Leuchte defekt") == "beleuchtung"
assert clean_text("Die Leuchten leuchten nicht") == ""  # Die Form "leuchten" bleibt unverändert.
print("Kontrolle der Textfilter: bestanden")



In [ ]:
# Schritt 10b Meldungen ohne Merkmale erfassen

def select_model_documents(documents, count_matrix, tfidf_matrix):
    """Die gleiche Auswahl von Meldungen auf beide Merkmalsmatrizen anwenden."""
    if len(documents) != count_matrix.shape[0] or count_matrix.shape != tfidf_matrix.shape:
        raise ValueError("Meldungen und Merkmalsmatrizen passen nicht zusammen.")
    if not documents.index.is_unique:
        raise ValueError("Der Index der Meldungen muss eindeutig sein.")

    count_matrix = count_matrix.tocsr(copy=True)
    tfidf_matrix = tfidf_matrix.tocsr(copy=True)
    for matrix in (count_matrix, tfidf_matrix):
        if not np.isfinite(matrix.data).all() or (matrix.data < 0).any():
            raise ValueError("Merkmalsmatrizen müssen endliche, nichtnegative Werte enthalten.")
        matrix.eliminate_zeros()

    count_mask = np.asarray(count_matrix.getnnz(axis=1)).ravel() > 0
    tfidf_mask = np.asarray(tfidf_matrix.getnnz(axis=1)).ravel() > 0
    if not np.array_equal(count_mask, tfidf_mask):
        raise ValueError("Count und TF-IDF ergeben unterschiedliche Meldungen ohne Merkmale.")

    # Die ursprünglichen Zeilenpositionen bleiben über den Index nachvollziehbar.
    fit_documents = documents.iloc[np.flatnonzero(count_mask)].copy()
    without_features = documents.iloc[np.flatnonzero(~count_mask)].copy()
    return (
        count_mask,
        fit_documents,
        count_matrix[count_mask],
        tfidf_matrix[count_mask],
        without_features,
    )


if not np.array_equal(
    count_vectorizer.get_feature_names_out(), tfidf_vectorizer.get_feature_names_out()
):
    raise ValueError("Count und TF-IDF verwenden unterschiedliche Vokabulare.")

model_mask, df_fit, X_count_fit, X_tfidf_fit, df_without_features = select_model_documents(
    df_model, X_count, X_tfidf
)
N_ANALYSIS_DOCS = len(df_model)
if N_ANALYSIS_DOCS == 0:
    raise ValueError("Nach dem Text Preprocessing sind keine Meldungen vorhanden.")

# Vokabular und IDF bleiben aus dem gesamten Korpus nach Text Preprocessing erhalten.
# Nur für das Training werden die Zeilen ohne Merkmale ausgeschlossen.
assert len(df_fit) + len(df_without_features) == N_ANALYSIS_DOCS
assert len(df_fit) == X_count_fit.shape[0] == X_tfidf_fit.shape[0]
assert (X_count_fit.getnnz(axis=1) > 0).all()
assert (X_tfidf_fit.getnnz(axis=1) > 0).all()

print("Meldungen nach Text Preprocessing:", N_ANALYSIS_DOCS)
print("Meldungen mit Merkmalen für das Modelltraining:", len(df_fit))
print("Nicht zugeordnet, da keine Merkmale im Vokabular:", len(df_without_features))
print("Count-Matrix für das Modelltraining:", X_count_fit.shape)
print("TF-IDF-Matrix für das Modelltraining:", X_tfidf_fit.shape)
print("Meldungen ohne Merkmale in den Trainingsmatrizen:", 0)
print("Bezugsgröße der Prozentanteile:", N_ANALYSIS_DOCS)
print("Kontrolle der Auswahl: bestanden")


In [ ]:
# Schritt 11 LDA und NMF trainieren

from sklearn.decomposition import LatentDirichletAllocation, NMF

if len(df_fit) == 0:
    raise ValueError("Keine Meldungen mit Merkmalen vorhanden. Daten und Vokabularfilter prüfen.")
if not isinstance(N_TOPICS, int) or isinstance(N_TOPICS, bool) or N_TOPICS < 1:
    raise ValueError("N_TOPICS muss eine positive ganze Zahl sein.")
if N_TOPICS > min(X_tfidf_fit.shape):
    raise ValueError("Für diese NMF-Initialisierung gibt es zu wenige Meldungen oder Merkmale.")

lda_model = LatentDirichletAllocation(
    n_components=N_TOPICS,
    random_state=RANDOM_STATE,
    learning_method="batch",
    doc_topic_prior=LDA_DOC_TOPIC_PRIOR,
    topic_word_prior=LDA_TOPIC_WORD_PRIOR,
    max_iter=LDA_MAX_ITER
)

nmf_model = NMF(
    n_components=N_TOPICS,
    random_state=RANDOM_STATE,
    init="nndsvda",
    max_iter=NMF_MAX_ITER
)

lda_doc_topic = lda_model.fit_transform(X_count_fit)
nmf_doc_topic_raw = nmf_model.fit_transform(X_tfidf_fit)

print("LDA Dokument-Topic-Matrix:", lda_doc_topic.shape)
print("NMF Dokument-Topic-Matrix:", nmf_doc_topic_raw.shape)


In [ ]:
# Schritt 11a NMF-Koeffizienten für einheitliche Topic-Vektoren berechnen

def normalize_nmf_for_assignment(coefficients, components):
    """Topic-Vektoren auf L2-Norm 1 bringen und die Koeffizienten entsprechend anpassen."""
    w = np.asarray(coefficients, dtype=float)
    h = np.asarray(components, dtype=float)
    if w.ndim != 2 or h.ndim != 2 or w.shape[1] != h.shape[0] or h.shape[0] == 0:
        raise ValueError("Die beiden NMF-Matrizen passen nicht zusammen.")
    if h.shape[1] == 0:
        raise ValueError("Die Topic-Vektoren enthalten keine Merkmale.")
    if any(not np.isfinite(a).all() or (a < 0).any() for a in (w, h)):
        raise ValueError("NMF-Koeffizienten müssen endlich und nicht negativ sein.")

    lengths = np.linalg.norm(h, axis=1)
    if not np.isfinite(lengths).all() or (lengths <= 0).any():
        raise ValueError("Mindestens ein Topic-Vektor hat keine verwendbare L2-Norm.")
    h_unit = h / lengths[:, None]
    w_scaled = w * lengths[None, :]
    if not np.isfinite(w_scaled).all():
        raise ValueError("Die angepassten NMF-Koeffizienten enthalten ungültige Werte.")
    if not np.allclose(np.linalg.norm(h_unit, axis=1), 1.0):
        raise RuntimeError("Die Topic-Vektoren haben nicht die erwartete L2-Norm.")

    # Das Produkt beider Matrizen muss erhalten bleiben, kleine Abweichungen in der Rundung sind zulässig.
    for start in range(0, len(w), 256):
        before = w[start:start + 256] @ h
        after = w_scaled[start:start + 256] @ h_unit
        if not np.allclose(before, after, rtol=1e-10, atol=1e-12):
            raise RuntimeError("Die Rekonstruktion wurde durch die Skalierung verändert.")
    return w_scaled, h_unit


# Gleiche Rekonstruktion bei unterschiedlicher Skalierung muss gleiche angepasste Werte ergeben.
_scale_test_w = np.array([[1.0, 2.0], [0.0, 0.0]])
_scale_test_h = np.array([[10.0, 0.0], [0.0, 1.0]])
_scale_test_a, _ = normalize_nmf_for_assignment(_scale_test_w, _scale_test_h)
_scale_test_factors = np.array([0.01, 100.0])
_scale_test_b, _ = normalize_nmf_for_assignment(
    _scale_test_w / _scale_test_factors,
    _scale_test_h * _scale_test_factors[:, None],
)
if not np.allclose(_scale_test_a, _scale_test_b) or np.any(_scale_test_a[1] != 0):
    raise RuntimeError("Der Funktionstest der NMF-Skalierung ist fehlgeschlagen.")

# Die ursprünglichen Komponenten und Koeffizienten bleiben unverändert erhalten.
nmf_doc_topic, nmf_components_unit = normalize_nmf_for_assignment(
    nmf_doc_topic_raw, nmf_model.components_
)
print("NMF-Zuordnung verwendet Koeffizienten für L2-normierte Topic-Vektoren.")
print("Technische Kontrolle der NMF-Skalierung: bestanden")


In [ ]:
# Schritt 11b Topic-Zuordnung und nicht zugeordnete Meldungen erfassen

def assign_topics(doc_topic_matrix, model_mask, source_index):
    """Modellgewichte den ursprünglichen Zeilen zuordnen und fehlende Merkmale kennzeichnen."""
    weights = np.asarray(doc_topic_matrix, dtype=float)
    mask = np.asarray(model_mask)
    if mask.ndim != 1 or mask.dtype != bool or len(mask) != len(source_index):
        raise ValueError("Die Zeilenauswahl passt nicht zum Index der Meldungen.")
    if not source_index.is_unique:
        raise ValueError("Der Index der Meldungen muss eindeutig sein.")
    if weights.ndim != 2 or weights.shape[0] != int(mask.sum()) or weights.shape[1] == 0:
        raise ValueError("Die Dokument-Topic-Matrix passt nicht zur Zeilenauswahl.")
    if not np.isfinite(weights).all() or (weights < 0).any():
        raise ValueError("Topic-Gewichte müssen endlich und nichtnegativ sein.")

    result = pd.DataFrame(index=source_index.copy())
    result["Topic"] = pd.Series(pd.NA, index=source_index, dtype="Int64")
    result["Status"] = pd.Series(
        "Keine Merkmale im Vokabular", index=source_index, dtype="string"
    )
    result["Gleicher Höchstwert"] = False
    positions = np.flatnonzero(mask)
    if len(positions) == 0:
        return result

    max_weights = weights.max(axis=1)
    positive = max_weights > 0
    valid_index = source_index.take(positions[positive])
    zero_index = source_index.take(positions[~positive])

    # Nur positive Gewichte für eine Zuordnung verwenden.
    # Bei exakt gleichen Höchstwerten wählt argmax die erste Topic-Nummer.
    labels = weights[positive].argmax(axis=1) + 1
    result.loc[valid_index, "Topic"] = labels
    result.loc[valid_index, "Status"] = "Topic zugeordnet"
    result.loc[zero_index, "Status"] = "Keine positiven Topic-Gewichte"
    tied = (weights[positive] == max_weights[positive, None]).sum(axis=1) > 1
    result.loc[valid_index, "Gleicher Höchstwert"] = tied
    return result


lda_assignments = assign_topics(lda_doc_topic, model_mask, df_model.index)
nmf_assignments = assign_topics(nmf_doc_topic, model_mask, df_model.index)

# Künstlicher Test: eine zugeordnete Meldung, eine ohne Merkmale und eine ohne positive Gewichte.
_test_assignments = assign_topics(
    np.array([[0.2, 0.8], [0.0, 0.0]]),
    np.array([True, False, True]),
    pd.Index([10, 20, 30]),
)
assert _test_assignments.loc[10, "Topic"] == 2
assert _test_assignments.loc[[20, 30], "Topic"].isna().all()
assert _test_assignments.loc[20, "Status"] == "Keine Merkmale im Vokabular"
assert _test_assignments.loc[30, "Status"] == "Keine positiven Topic-Gewichte"

status_rows = []
for method, assignments in (("LDA", lda_assignments), ("NMF", nmf_assignments)):
    if not assignments.index.equals(df_model.index):
        raise RuntimeError("Die Reihenfolge der Zuordnungen passt nicht zu den Meldungen.")
    if assignments.loc[~model_mask, "Topic"].notna().any():
        raise RuntimeError("Eine Meldung ohne Merkmale hat unzulässigerweise ein Topic erhalten.")
    assigned_count = int(assignments["Topic"].notna().sum())
    unassigned_count = int(assignments["Topic"].isna().sum())
    assert assigned_count + unassigned_count == N_ANALYSIS_DOCS
    status_rows.append({
        "Verfahren": method,
        "Meldungen nach Text Preprocessing": N_ANALYSIS_DOCS,
        "Topic zugeordnet": assigned_count,
        "Nicht zugeordnet": unassigned_count,
        "Davon ohne Merkmale": int((assignments["Status"] == "Keine Merkmale im Vokabular").sum()),
        "Davon ohne positive Topic-Gewichte": int((assignments["Status"] == "Keine positiven Topic-Gewichte").sum()),
        "Anteil nicht zugeordnet in Prozent": round(100 * unassigned_count / N_ANALYSIS_DOCS, 1),
        "Mehrere Topics mit gleichem Höchstwert": int(assignments["Gleicher Höchstwert"].sum()),
    })
    print(f"{method}: {assigned_count} zugeordnet, {unassigned_count} nicht zugeordnet")

assignment_summary = pd.DataFrame(status_rows)
display(assignment_summary)
print("Kontrolle der Zuordnung: bestanden")


## Aktueller Modellvergleich

Der Vergleich verwendet die ergänzte Zuordnung `leuchte → beleuchtung` und bei allen NMF-Modellen dieselbe L2-Skalierung für die Zuordnung.
Kohärenzwerte und Meldungshäufigkeiten werden neu berechnet. Die endgültige Modellwahl erfolgt nach der inhaltlichen Prüfung der zugeordneten Originalmeldungen.


In [ ]:
# Schritt 11d verschiedene Anzahl von Topics vergleichen

import warnings
from pathlib import Path

import gensim
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
from sklearn.base import clone
from sklearn.exceptions import ConvergenceWarning

# Diese Einstellungen gelten nur für den Vergleich
TOPIC_COUNTS_TO_COMPARE = [6, 8, 10, 12, 15]
COHERENCE_TOP_N = 10
COHERENCE_WINDOW_SIZE = 110


def compare_topic_counts(
    documents, fit_documents, count_matrix, tfidf_matrix, selected_rows,
    count_features, tfidf_features, lda_template, nmf_template,
    topic_counts, top_n, window_size,
):
    """Topic-Modelle auf derselben Datengrundlage trainieren und mit c_v vergleichen."""
    if count_matrix.shape != tfidf_matrix.shape:
        raise ValueError("Count und TF-IDF müssen dieselbe Form haben.")
    if len(fit_documents) != count_matrix.shape[0] or len(fit_documents) == 0:
        raise ValueError("Die Merkmalsmatrizen passen nicht zu den Meldungen für das Training.")
    if not np.array_equal(count_features, tfidf_features):
        raise ValueError("Count und TF-IDF müssen dasselbe Vokabular verwenden.")
    if len(count_features) != count_matrix.shape[1]:
        raise ValueError("Das Vokabular passt nicht zur Zahl der Merkmale.")
    mask = np.asarray(selected_rows)
    if mask.ndim != 1 or mask.dtype != bool or len(mask) != len(documents):
        raise ValueError("Die Auswahl der Meldungen ist ungültig.")
    if not documents.index.is_unique or not documents.index[mask].equals(fit_documents.index):
        raise ValueError("Die Reihenfolge der Meldungen stimmt nicht mit der Auswahl überein.")
    if not documents.loc[mask, "text_clean"].equals(fit_documents["text_clean"]):
        raise ValueError("Die Texte für das Training stimmen nicht mit der Auswahl überein.")
    for matrix in (count_matrix, tfidf_matrix):
        if not np.isfinite(matrix.data).all() or (matrix.data < 0).any():
            raise ValueError("Die Matrizen müssen endliche, nicht negative Werte enthalten.")
        if (np.asarray((matrix != 0).sum(axis=1)).ravel() == 0).any():
            raise ValueError("Die Trainingsmatrizen enthalten Meldungen ohne Merkmale.")
    if not topic_counts or len(set(topic_counts)) != len(topic_counts):
        raise ValueError("Die Anzahl der Topics muss eindeutig vorgegeben sein.")
    if any(isinstance(k, bool) or not isinstance(k, int) or k < 2 for k in topic_counts):
        raise ValueError("Für jeden Versuch sind mindestens zwei Topics erforderlich.")
    if max(topic_counts) > min(tfidf_matrix.shape):
        raise ValueError("Für die NMF-Initialisierung gibt es zu wenige Meldungen oder Merkmale.")
    if isinstance(top_n, bool) or not isinstance(top_n, int) or not 2 <= top_n <= len(count_features):
        raise ValueError("Die Anzahl der Top-Begriffe passt nicht zum Vokabular.")
    if isinstance(window_size, bool) or not isinstance(window_size, int) or window_size < 2:
        raise ValueError("Die Fenstergröße für c_v muss mindestens zwei betragen.")
    if tuple(count_vectorizer.ngram_range) != (1, 1):
        raise ValueError("Dieser Kohärenzvergleich ist für die aktuelle Unigramm-Darstellung ausgelegt.")

    # Alle Varianten verwenden dieselben bereinigten Texte für die Kohärenzberechnung.
    reference_texts = documents["text_clean"].str.split().tolist()
    if any(not isinstance(text, list) or not text for text in reference_texts):
        raise ValueError("Die Referenztexte für c_v dürfen nicht fehlen oder leer sein.")
    dictionary = Dictionary(reference_texts)
    missing_terms = set(count_features) - set(dictionary.token2id)
    if missing_terms:
        raise ValueError("Begriffe aus dem Vokabular fehlen in den Referenztexten für c_v.")

    # Die effektiven LDA-Priors bleiben wie im bisherigen Modell, statt sich mit 1/k zu ändern.
    lda_priors = {
        "doc_topic_prior": float(lda_template.doc_topic_prior_),
        "topic_word_prior": float(lda_template.topic_word_prior_),
    }
    if any(not np.isfinite(value) or value <= 0 for value in lda_priors.values()):
        raise ValueError("Die LDA-Priors müssen positiv und endlich sein.")

    models, weights_by_model, assignments_by_model = {}, {}, {}
    summary_rows, topic_rows = [], []
    print("Meldungen für c_v:", len(documents))
    print("Meldungen je Modelltraining:", len(fit_documents))
    print("Merkmale:", count_matrix.shape[1])
    print("Anzahl der Topics:", topic_counts)
    print("Top-Begriffe je Topic für c_v:", top_n)
    print("Feste LDA-Priors:", lda_priors)

    total_runs = 2 * len(topic_counts)
    completed = 0
    for method, template, matrix in (
        ("LDA", lda_template, count_matrix),
        ("NMF", nmf_template, tfidf_matrix),
    ):
        for k in topic_counts:
            print(f"[{completed + 1}/{total_runs}] {method} mit {k} Topics", flush=True)
            candidate = clone(template).set_params(n_components=k)
            if method == "LDA":
                candidate.set_params(**lda_priors)
            with warnings.catch_warnings(record=True) as recorded_warnings:
                warnings.simplefilter("always", ConvergenceWarning)
                weights = candidate.fit_transform(matrix)
            warning_messages = [str(warning.message) for warning in recorded_warnings]
            for message in warning_messages:
                print(f"  Warnung: {message}")

            components = candidate.components_
            if components.shape != (k, len(count_features)):
                raise RuntimeError("Die Modell-Komponenten passen nicht zum Vergleich.")
            if not np.isfinite(components).all() or (components < 0).any():
                raise RuntimeError("Das Modell enthält ungültige Gewichte.")
            if ((components > 0).sum(axis=1) < top_n).any():
                raise RuntimeError("Ein Topic enthält zu wenige positive Gewichte für die Top-Begriffe.")
            top_indices = np.argsort(-components, axis=1, kind="stable")[:, :top_n]
            topics = [count_features[indices].tolist() for indices in top_indices]

            # Ein Prozess vermeidet zusätzliche Worker bei der Ausführung unter Windows.
            coherence = CoherenceModel(
                topics=topics, texts=reference_texts, dictionary=dictionary,
                coherence="c_v", topn=top_n, window_size=window_size, processes=1,
            )
            per_topic = np.asarray(coherence.get_coherence_per_topic(), dtype=float)
            if per_topic.shape != (k,) or not np.isfinite(per_topic).all():
                raise RuntimeError(f"Für {method} mit {k} Topics ist c_v nicht vollständig berechenbar.")

            # Alle NMF-Varianten verwenden dieselbe Skalierung für die Zuordnung.
            # Das Modell und die für c_v verwendeten Top-Begriffe bleiben dabei unverändert.
            if method == "NMF":
                assignment_weights, _ = normalize_nmf_for_assignment(weights, components)
                assignment_rule = "NMF mit L2-normierten Topic-Vektoren"
            else:
                assignment_weights = weights
                assignment_rule = "LDA-Themenanteile"
            assignments = assign_topics(assignment_weights, mask, documents.index)
            if assignments.loc[~mask, "Topic"].notna().any():
                raise RuntimeError("Eine Meldung ohne Merkmale wurde einem Topic zugeordnet.")
            counts = assignments["Topic"].value_counts().reindex(range(1, k + 1), fill_value=0)
            assigned = int(assignments["Topic"].notna().sum())
            unassigned = int(assignments["Topic"].isna().sum())
            if int(counts.sum()) != assigned or assigned + unassigned != len(documents):
                raise RuntimeError("Die Häufigkeiten des Vergleichs sind nicht konsistent.")

            key = (method, k)
            models[key] = candidate
            # Diese Werte müssen zu den gespeicherten Topic-Zuordnungen passen.
            weights_by_model[key] = assignment_weights
            assignments_by_model[key] = assignments
            summary_rows.append({
                "Verfahren": method,
                "Anzahl Topics": k,
                "c_v Mittelwert": float(per_topic.mean()),
                "c_v Minimum": float(per_topic.min()),
                "Zugeordnet": assigned,
                "Nicht zugeordnet": unassigned,
                "Topics ohne Zuordnung": int(counts.eq(0).sum()),
                "Iterationen": int(candidate.n_iter_),
                "Warnungen": len(warning_messages),
                "Hinweise": " | ".join(warning_messages),
                "Regel der Zuordnung": assignment_rule,
            })
            for position, (terms, value) in enumerate(zip(topics, per_topic), start=1):
                topic_rows.append({
                    "Verfahren": method,
                    "Anzahl Topics": k,
                    "Topic": position,
                    "Anzahl Meldungen": int(counts.loc[position]),
                    "Anteil Prozent": 100 * int(counts.loc[position]) / len(documents),
                    "Bezugsmenge Meldungen": len(documents),
                    "c_v": float(value),
                    "Top-Begriffe": ", ".join(terms),
                    "Regel der Zuordnung": assignment_rule,
                })
            completed += 1
            print(f"  c_v Mittelwert: {per_topic.mean():.4f}, nicht zugeordnet: {unassigned}", flush=True)

    return (
        pd.DataFrame(summary_rows), pd.DataFrame(topic_rows),
        models, weights_by_model, assignments_by_model,
    )


required_for_topic_comparison = (
    "df_model", "df_fit", "model_mask", "X_count_fit", "X_tfidf_fit",
    "count_vectorizer", "tfidf_vectorizer", "lda_model", "nmf_model",
    "assign_topics", "normalize_nmf_for_assignment", "OUTPUT_DIR",
)
missing_for_topic_comparison = [
    name for name in required_for_topic_comparison if name not in globals()
]
if missing_for_topic_comparison:
    raise RuntimeError(
        "Zuerst alle Zellen bis einschließlich Schritt 11b ausführen. "
        f"Es fehlen: {', '.join(missing_for_topic_comparison)}"
    )

(
    topic_count_review, topic_count_details,
    topic_count_models, topic_count_weights, topic_count_assignments,
) = compare_topic_counts(
    df_model, df_fit, X_count_fit, X_tfidf_fit, model_mask,
    count_vectorizer.get_feature_names_out(), tfidf_vectorizer.get_feature_names_out(),
    lda_model, nmf_model, TOPIC_COUNTS_TO_COMPARE, COHERENCE_TOP_N, COHERENCE_WINDOW_SIZE,
)

print("\nVergleich der Anzahl von Topics")
display(topic_count_review.drop(columns=["Hinweise"]).round(4))

# Nur Vergleichswerte und Top-Begriffe exportieren, keine originalen Meldungen.
comparison_dir = Path(OUTPUT_DIR)
comparison_dir.mkdir(parents=True, exist_ok=True)
comparison_path = comparison_dir / "vergleich_topic_anzahl.csv"
comparison_topics_path = comparison_dir / "vergleich_topic_anzahl_details.csv"
topic_count_review.to_csv(comparison_path, index=False, sep=";", encoding="utf-8-sig")
topic_count_details.to_csv(comparison_topics_path, index=False, sep=";", encoding="utf-8-sig")

print("Gespeichert:")
print(comparison_path.resolve())
print(comparison_topics_path.resolve())
print("Technische Kontrolle des Vergleichs: bestanden")
print("Die Modelle und Zuordnungen der Hauptanalyse sind unverändert.")

# Einstellungen mit den Ergebnissen speichern, damit unterschiedliche Läufe erkennbar bleiben.
import hashlib
import json
import sklearn

comparison_settings = {
    "csv_sha256": hashlib.sha256(csv_path.read_bytes()).hexdigest(),
    "leuchte_zuordnung": CANONICAL_TERMS_FINAL.get("leuchte"),
    "nmf_zuordnung": "L2-normierte Topic-Vektoren",
    "anzahl_topics": list(TOPIC_COUNTS_TO_COMPARE),
    "anzahl_meldungen_nach_preprocessing": len(df_model),
    "anzahl_meldungen_training": len(df_fit),
    "anzahl_merkmale": X_count_fit.shape[1],
    "min_df": MIN_DF,
    "max_df": MAX_DF,
    "ngram_range": list(NGRAM_RANGE),
    "token_pattern": TOKEN_PATTERN,
    "sublinear_tf": SUBLINEAR_TF,
    "min_words_raw": MIN_WORDS_RAW,
    "min_tokens_clean": MIN_TOKENS_CLEAN,
    "min_token_length": MIN_TOKEN_LENGTH,
    "stopwoerter": sorted(STOPWORDS_FINAL),
    "begriffszuordnungen": dict(CANONICAL_TERMS_FINAL),
    "random_state": RANDOM_STATE,
    "lda_doc_topic_prior": LDA_DOC_TOPIC_PRIOR,
    "lda_topic_word_prior": LDA_TOPIC_WORD_PRIOR,
    "lda_max_iter": LDA_MAX_ITER,
    "nmf_max_iter": NMF_MAX_ITER,
    "coherence": "c_v",
    "coherence_top_n": COHERENCE_TOP_N,
    "coherence_window_size": COHERENCE_WINDOW_SIZE,
    "gensim": gensim.__version__,
    "scikit_learn": sklearn.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
}
settings_path = comparison_dir / "vergleich_topic_anzahl_einstellungen.json"
settings_path.write_text(json.dumps(comparison_settings, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


In [ ]:
# Schritt 11e Meldungen ausgewählter Modelle zur inhaltlichen Prüfung zusammenstellen

from pathlib import Path

import numpy as np
import pandas as pd

# Die Kandidaten werden nur geprüft, die Hauptanalyse bleibt unverändert.
MODELS_TO_REVIEW = [("NMF", 12), ("NMF", 15), ("LDA", 10), ("LDA", 15)]
EXAMPLES_BY_WEIGHT = 2
RANDOM_EXAMPLES = 3


def collect_topic_examples(
    documents, fit_documents, selected_rows, models, weights_by_model,
    assignments_by_model, details, candidates, text_column, id_column,
    n_by_weight=2, n_random=3, random_state=42,
):
    """Pro Topic die stark gewichteten Meldungen und eine getrennte Zufallsauswahl zusammenstellen."""
    required_columns = {text_column, id_column, "category_parent", "category_child"}
    if not required_columns.issubset(documents.columns):
        raise ValueError("Textspalte, ID oder Kategorien fehlen in den Meldungen.")
    if not documents.index.is_unique or not fit_documents.index.is_unique:
        raise ValueError("Die Indizes der Meldungen müssen eindeutig sein.")
    if documents[id_column].isna().any() or documents[id_column].duplicated().any():
        raise ValueError("Die Melde-IDs müssen vorhanden und eindeutig sein.")
    if any(isinstance(n, bool) or not isinstance(n, int) or n < 0 for n in (n_by_weight, n_random)):
        raise ValueError("Die Anzahl der Beispiele muss jeweils eine nicht negative ganze Zahl sein.")
    if n_by_weight + n_random == 0:
        raise ValueError("Mindestens eine Meldung pro Topic muss ausgewählt werden.")
    if not candidates or len(set(candidates)) != len(candidates):
        raise ValueError("Die Liste der zu prüfenden Modelle darf nicht leer sein oder Einträge wiederholen.")

    mask = np.asarray(selected_rows)
    if mask.ndim != 1 or mask.dtype != bool or len(mask) != len(documents):
        raise ValueError("Die Auswahl der Meldungen passt nicht zum Datensatz.")
    if not documents.index[mask].equals(fit_documents.index):
        raise ValueError("Die Reihenfolge der Meldungen stimmt nicht mit den Modellgewichten überein.")
    if not documents.loc[mask, "text_clean"].equals(fit_documents["text_clean"]):
        raise ValueError("Die Texte wurden seit dem Modellvergleich verändert.")

    rng = np.random.default_rng(random_state)
    rows = []
    overview = []
    fit_index = fit_documents.index

    for method, k in candidates:
        key = (method, k)
        if any(key not in collection for collection in (models, weights_by_model, assignments_by_model)):
            raise ValueError(f"Das Vergleichsmodell {method} mit {k} Topics fehlt. Zuerst Schritt 11d ausführen.")
        model = models[key]
        # Bei NMF sind dies die bereits angepassten Koeffizienten aus Schritt 11d.
        weights = np.asarray(weights_by_model[key], dtype=float)
        assignments = assignments_by_model[key]
        if model.n_components != k or weights.shape != (len(fit_documents), k):
            raise ValueError(f"Die Modellgewichte von {method} mit {k} Topics haben eine unerwartete Form.")
        if not np.isfinite(weights).all() or (weights < 0).any():
            raise ValueError("Die Modellgewichte müssen endlich und nicht negativ sein.")
        if not assignments.index.equals(documents.index):
            raise ValueError("Die Zuordnungen passen nicht zum Index der Meldungen.")
        if assignments.loc[~mask, "Topic"].notna().any():
            raise ValueError("Eine Meldung ohne Merkmale hat eine Topic-Zuordnung.")

        # Die gespeicherten Zuordnungen müssen zu den vorhandenen Gewichtungen passen.
        expected = pd.Series(pd.NA, index=fit_index, dtype="Int64")
        positive = weights.max(axis=1) > 0
        expected.loc[fit_index[positive]] = weights[positive].argmax(axis=1) + 1
        actual = assignments.loc[fit_index, "Topic"].astype("Int64")
        if not np.array_equal(expected.to_numpy(na_value=-1), actual.to_numpy(na_value=-1)):
            raise ValueError("Modellgewichte und Topic-Zuordnungen stimmen nicht überein.")

        model_details = details.loc[
            (details["Verfahren"] == method) & (details["Anzahl Topics"] == k)
        ].set_index("Topic")
        if not model_details.index.is_unique or set(model_details.index) != set(range(1, k + 1)):
            raise ValueError("Die Detailtabelle enthält nicht genau einen Eintrag pro Topic.")

        before_model = len(rows)
        for topic in range(1, k + 1):
            positions = np.flatnonzero(actual.eq(topic).fillna(False).to_numpy(dtype=bool))
            detail = model_details.loc[topic]
            if len(positions) != int(detail["Anzahl Meldungen"]):
                raise ValueError("Die Häufigkeiten der Detailtabelle passen nicht zu den Zuordnungen.")

            # Hohe Gewichtungen dienen nur zur Auswahl von Beispielen, nicht als Beweis einer korrekten Zuordnung.
            order = np.argsort(-weights[positions, topic - 1], kind="stable")
            strongest = positions[order[:min(n_by_weight, len(positions))]]
            remaining = positions[~np.isin(positions, strongest)]
            random_selection = rng.choice(remaining, size=min(n_random, len(remaining)), replace=False)
            selections = [(int(p), "Hohes Topic-Gewicht") for p in strongest]
            selections += [(int(p), "Zufallsauswahl") for p in random_selection]
            if len({p for p, _ in selections}) != len(selections):
                raise RuntimeError("Eine Meldung wurde innerhalb eines Topics mehrfach ausgewählt.")

            for position, selection in selections:
                record = documents.loc[fit_index[position]]
                rows.append({
                    "Verfahren": method,
                    "Anzahl Topics": k,
                    "Topic": topic,
                    "Top-Begriffe": detail["Top-Begriffe"],
                    "c_v": float(detail["c_v"]),
                    "Meldungen im Topic": len(positions),
                    "Auswahl": selection,
                    "ID": record[id_column],
                    "Meldetext": record[text_column],
                    "Hauptkategorie": record["category_parent"],
                    "Unterkategorie": record["category_child"],
                })
        overview.append({"Verfahren": method, "Anzahl Topics": k, "Ausgewählte Meldungen": len(rows) - before_model})

    columns = [
        "Verfahren", "Anzahl Topics", "Topic", "Top-Begriffe", "c_v",
        "Meldungen im Topic", "Auswahl", "ID", "Meldetext", "Hauptkategorie", "Unterkategorie",
    ]
    return pd.DataFrame(rows, columns=columns), pd.DataFrame(overview)


required_for_examples = (
    "df_model", "df_fit", "model_mask", "topic_count_models", "topic_count_weights",
    "topic_count_assignments", "topic_count_details", "TEXT_COLUMN", "ID_COLUMN", "OUTPUT_DIR",
)
missing_for_examples = [name for name in required_for_examples if name not in globals()]
if missing_for_examples:
    raise RuntimeError(
        "Zuerst die Hauptanalyse bis Schritt 11b und den Modellvergleich in Schritt 11d ausführen. "
        f"Es fehlen: {', '.join(missing_for_examples)}"
    )

review_topic_examples, review_example_counts = collect_topic_examples(
    df_model, df_fit, model_mask,
    topic_count_models, topic_count_weights, topic_count_assignments, topic_count_details,
    MODELS_TO_REVIEW, TEXT_COLUMN, ID_COLUMN,
    n_by_weight=EXAMPLES_BY_WEIGHT, n_random=RANDOM_EXAMPLES, random_state=RANDOM_STATE,
)

# Die Datei enthält die Originalmeldungen und bleibt im lokalen Ausgabeordner.
example_path = Path(OUTPUT_DIR) / "pruefung_topic_meldungen.csv"
example_path.parent.mkdir(parents=True, exist_ok=True)
review_topic_examples.to_csv(example_path, index=False, sep=";", encoding="utf-8-sig")
print(review_example_counts.to_string(index=False))
print("Insgesamt ausgewählte Meldungen:", len(review_topic_examples))
print("Technische Kontrolle der Auswahl: bestanden")
print("Gespeichert:", example_path.resolve())


## Bewertung des aktuellen Modellvergleichs

NMF mit zwölf Topics erzielt den höchsten mittleren Kohärenzwert der
untersuchten Varianten (`c_v = 0,4201`) und wird für die weitere
Ergebnisdarstellung bevorzugt. Innerhalb der LDA-Varianten liegt
das Modell mit zehn Topics vorn (`c_v = 0,3705`).

Die Prüfung ausgeworiginaler Meldungenldungen zeigt klar erkennbare
Schwerpunkte, aber auch gemischte Topics und einzelne unpassende
Zuordnungen.

In [ ]:
# Schritt 11f Ausgewählte Modelle für die Ergebnisübersichten übernehmen

required_for_selection = (
    "SELECTED_TOPIC_COUNTS", "topic_count_models", "topic_count_weights",
    "topic_count_assignments", "topic_count_review", "topic_count_details",
    "df_model", "df_fit", "model_mask", "assign_topics", "OUTPUT_DIR",
)
missing_for_selection = [name for name in required_for_selection if name not in globals()]
if missing_for_selection:
    raise RuntimeError(
        "Zuerst die Konfiguration und den Modellvergleich ausführen. "
        f"Es fehlen: {', '.join(missing_for_selection)}"
    )
if set(SELECTED_TOPIC_COUNTS) != {"LDA", "NMF"}:
    raise ValueError("Für LDA und NMF muss jeweils eine Anzahl von Topics gewählt sein.")

selected_models, selected_weights, selected_assignments = {}, {}, {}
selection_rows, selected_status_rows = [], []

for method in ("LDA", "NMF"):
    k = SELECTED_TOPIC_COUNTS[method]
    if isinstance(k, bool) or not isinstance(k, int) or k < 2:
        raise ValueError("Die gewählte Anzahl von Topics muss eine ganze Zahl ab zwei sein.")
    key = (method, k)
    if any(key not in collection for collection in (
        topic_count_models, topic_count_weights, topic_count_assignments
    )):
        raise ValueError(f"Das Vergleichsmodell {method} mit {k} Topics fehlt.")

    model = topic_count_models[key]
    coefficients = np.asarray(topic_count_weights[key], dtype=float).copy()
    assignments = topic_count_assignments[key].copy(deep=True)
    if model.n_components != k or coefficients.shape != (len(df_fit), k):
        raise ValueError("Das Modell und die Topic-Matrix passen nicht zusammen.")

    components = np.asarray(model.components_, dtype=float)
    if (not np.isfinite(components).all() or (components < 0).any()
            or (np.linalg.norm(components, axis=1) <= 0).any()):
        raise ValueError("Die ausgewählten Topic-Vektoren enthalten ungültige Werte.")

    # Die Auswahl übernimmt genau die bereits geprüften Zuordnungen.
    expected_assignments = assign_topics(coefficients, model_mask, df_model.index)
    if not assignments.equals(expected_assignments):
        raise ValueError("Die gespeicherten Zuordnungen passen nicht zu den Koeffizienten.")

    result = topic_count_review.loc[
        (topic_count_review["Verfahren"] == method)
        & (topic_count_review["Anzahl Topics"] == k)
    ]
    details = topic_count_details.loc[
        (topic_count_details["Verfahren"] == method)
        & (topic_count_details["Anzahl Topics"] == k)
    ].set_index("Topic")
    if len(result) != 1 or not details.index.is_unique or set(details.index) != set(range(1, k + 1)):
        raise ValueError("Der gewählte Modellstand ist in den Vergleichstabellen nicht eindeutig vorhanden.")
    counts = assignments["Topic"].value_counts().reindex(range(1, k + 1), fill_value=0)
    if not np.array_equal(counts.to_numpy(), details.loc[range(1, k + 1), "Anzahl Meldungen"].to_numpy()):
        raise ValueError("Die Häufigkeiten stimmen nicht mit dem Modellvergleich überein.")

    assigned = int(assignments["Topic"].notna().sum())
    unassigned = int(assignments["Topic"].isna().sum())
    if assigned + unassigned != len(df_model) or assigned != int(result.iloc[0]["Zugeordnet"]):
        raise ValueError("Die Summen der Zuordnungen stimmen nicht mit dem Modellvergleich überein.")

    selected_models[method] = model
    selected_weights[method] = coefficients
    selected_assignments[method] = assignments
    selection_rows.append({
        "Verfahren": method, "Anzahl Topics": k,
        "c_v Mittelwert": float(result.iloc[0]["c_v Mittelwert"]),
        "Zugeordnet": assigned, "Nicht zugeordnet": unassigned,
    })
    selected_status_rows.append({
        "Verfahren": method, "Anzahl Topics": k,
        "Meldungen nach Text Preprocessing": len(df_model),
        "Topic zugeordnet": assigned, "Nicht zugeordnet": unassigned,
        "Davon ohne Merkmale": int(assignments["Status"].eq("Keine Merkmale im Vokabular").sum()),
        "Davon ohne positive Topic-Gewichte": int(assignments["Status"].eq("Keine positiven Topic-Gewichte").sum()),
        "Anteil von nicht zugeordnet in Prozent": round(100 * unassigned / len(df_model), 1),
        "Mehrere Topics mit gleichem Höchstwert": int(assignments["Gleicher Höchstwert"].sum()),
    })

# Alle nachfolgenden Ausgaben verwenden diese Modelle und die dazugehörigen Zuordnungen.

lda_model, nmf_model = selected_models["LDA"], selected_models["NMF"]
lda_doc_topic, nmf_doc_topic = selected_weights["LDA"], selected_weights["NMF"]
lda_assignments, nmf_assignments = selected_assignments["LDA"], selected_assignments["NMF"]

# Die NMF-Koeffizienten aus dem Vergleich sind bereits auf L2-normierte Topics abgestimmt.
_nmf_lengths = np.linalg.norm(nmf_model.components_, axis=1)
nmf_components_unit = nmf_model.components_ / _nmf_lengths[:, None]
nmf_doc_topic_raw = nmf_doc_topic / _nmf_lengths[None, :]

N_ANALYSIS_DOCS = len(df_model)
assignment_summary = pd.DataFrame(selected_status_rows)
model_selection = pd.DataFrame(selection_rows)

print(model_selection.round({"c_v Mittelwert": 4}).to_string(index=False))
print("Bezugsgröße der Prozentanteile:", N_ANALYSIS_DOCS)
print("Technische Kontrolle der Modellauswahl: bestanden")

selection_path = OUTPUT_DIR / "modellauswahl.csv"
model_selection.to_csv(selection_path, index=False, sep=";", encoding="utf-8-sig")
print("Gespeichert:", selection_path.resolve())


In [ ]:
# Schritt 11g LDA und NMF anhand derselben Meldungen vergleichen
# Die Zelle stellt eine Stichprobe bereit

from pathlib import Path
import numpy as np
import pandas as pd

REVIEW_SAMPLE_SIZE = 100
REVIEW_RANDOM_STATE = 42


def build_shared_review(
    documents: pd.DataFrame,
    models: dict,
    assignments: dict,
    vocabularies: dict,
    text_column: str,
    id_column: str,
    sample_size: int = 100,
    random_state: int = 42,
):
    """Eine gemeinsame, zufällige Stichprobe mit den vorhandenen Zuordnungen verbinden."""
    if not documents.columns.is_unique or not documents.index.is_unique:
        raise ValueError("Spaltennamen und Index der Meldungen müssen eindeutig sein.")
    if text_column == id_column or not {text_column, id_column}.issubset(documents.columns):
        raise ValueError("Textspalte und ID-Spalte fehlen oder sind nicht unterschiedlich.")
    if isinstance(sample_size, bool) or not isinstance(sample_size, int):
        raise ValueError("Die Größe der Stichprobe muss eine ganze Zahl sein.")
    if not 1 <= sample_size <= len(documents):
        raise ValueError("Die Größe der Stichprobe muss zwischen 1 und der Anzahl der Meldungen liegen.")

    ids = documents[id_column].astype("string").str.strip()
    if ids.isna().any() or ids.eq("").any() or ids.duplicated().any():
        raise ValueError("Die Melde-IDs müssen vorhanden und eindeutig sein.")
    texts = documents[text_column]
    if texts.isna().any() or texts.astype("string").str.strip().eq("").any():
        raise ValueError("Die Stichprobe benötigt nicht leere originale Meldungen.")
    if any(set(items) != {"LDA", "NMF"} for items in (models, assignments, vocabularies)):
        raise ValueError("Für LDA und NMF müssen Modell, Zuordnungen und Vokabular vorhanden sein.")

    # Sortierte IDs und ein fester Seed halten sie bei unverändertem Datenbestand reproduzierbar.
    population = documents[[id_column, text_column]].copy()
    population.columns = ["ID", "Meldetext"]
    population["ID"] = ids
    sample = population.sort_values("ID", kind="stable").sample(
        n=sample_size, replace=False, random_state=random_state
    )
    sample.insert(0, "Prüffall", np.arange(1, sample_size + 1))
    text_review = sample.reset_index(drop=True)
    comparison = text_review.copy(deep=True)
    overview = []

    for method in ("LDA", "NMF"):
        model = models[method]
        topic_assignments = assignments[method]
        vocabulary = np.asarray(vocabularies[method])
        components = np.asarray(model.components_, dtype=float)
        k = model.n_components

        if not {"Topic", "Status"}.issubset(topic_assignments.columns):
            raise ValueError(f"Topic oder Status fehlen bei {method}.")
        if not topic_assignments.index.equals(documents.index):
            raise ValueError(f"Die Reihenfolge der Zuordnungen passt bei {method} nicht zu den Meldungen.")
        if vocabulary.ndim != 1 or len(set(vocabulary)) != len(vocabulary):
            raise ValueError(f"Das Vokabular von {method} ist nicht eindeutig.")
        if components.shape != (k, len(vocabulary)):
            raise ValueError(f"Die Topic-Vektoren passen bei {method} nicht zum Vokabular.")
        if not np.isfinite(components).all() or (components < 0).any():
            raise ValueError(f"Die Topic-Vektoren von {method} enthalten ungültige Werte.")
        if not topic_assignments["Topic"].dropna().isin(range(1, k + 1)).all():
            raise ValueError(f"Die Topic-Zuordnungen von {method} enthalten unbekannte Nummern.")
        if topic_assignments["Status"].isna().any():
            raise ValueError(f"Bei {method} fehlt ein Status.")
        if not np.array_equal(
            topic_assignments["Status"].eq("Topic zugeordnet").to_numpy(dtype=bool),
            topic_assignments["Topic"].notna().to_numpy(),
        ):
            raise ValueError(f"Topic-Zuordnung und Status widersprechen sich bei {method}.")

        top_terms = {
            topic + 1: ", ".join(
                vocabulary[np.argsort(-components[topic], kind="stable")[:10]]
            )
            for topic in range(k)
        }
        chosen = topic_assignments.loc[sample.index, ["Topic", "Status"]].reset_index(drop=True)
        comparison[f"{method} Anzahl Topics"] = k
        comparison[f"{method} Topic"] = chosen["Topic"].astype("Int64")
        comparison[f"{method} Status"] = chosen["Status"]
        comparison[f"{method} Top-Begriffe"] = chosen["Topic"].map(top_terms).fillna("")
        overview.append({
            "Verfahren": method,
            "Anzahl Topics": k,
            "Meldungen in der Stichprobe": len(chosen),
            "Mit Topic-Zuordnung": int(chosen["Topic"].notna().sum()),
            "Ohne Topic-Zuordnung": int(chosen["Topic"].isna().sum()),
        })

    if len(comparison) != sample_size or comparison["ID"].duplicated().any():
        raise RuntimeError("Die Stichprobe hat nicht den erwarteten Umfang oder enthält doppelte IDs.")
    if not comparison[["Prüffall", "ID", "Meldetext"]].equals(text_review):
        raise RuntimeError("Die beiden Dateien beziehen sich nicht auf dieselben Meldungen.")
    return text_review, comparison, pd.DataFrame(overview)


_required_shared_review = (
    "df_model", "lda_model", "nmf_model", "lda_assignments", "nmf_assignments",
    "count_vectorizer", "tfidf_vectorizer", "SELECTED_TOPIC_COUNTS",
    "TEXT_COLUMN", "ID_COLUMN", "OUTPUT_DIR",
)
_missing_shared_review = [name for name in _required_shared_review if name not in globals()]
if _missing_shared_review:
    raise RuntimeError("Zuerst bis Schritt 11f ausführen. Es fehlen: " + ", ".join(_missing_shared_review))
if {"LDA": lda_model.n_components, "NMF": nmf_model.n_components} != SELECTED_TOPIC_COUNTS:
    raise ValueError("Die aktiven Modelle entsprechen nicht der Auswahl aus Schritt 11f.")

shared_review_texts, shared_review_comparison, shared_review_counts = build_shared_review(
    df_model,
    {"LDA": lda_model, "NMF": nmf_model},
    {"LDA": lda_assignments, "NMF": nmf_assignments},
    {
        "LDA": count_vectorizer.get_feature_names_out(),
        "NMF": tfidf_vectorizer.get_feature_names_out(),
    },
    TEXT_COLUMN, ID_COLUMN,
    sample_size=REVIEW_SAMPLE_SIZE, random_state=REVIEW_RANDOM_STATE,
)

# Die erste Datei enthält nur Meldetexte und IDs für die Beurteilung vor dem Modellvergleich.
# Beide Dateien enthalten originale Meldungen und bleiben lokal, Bewertungen separat speichern.
_shared_dir = Path(OUTPUT_DIR)
_shared_dir.mkdir(parents=True, exist_ok=True)
_shared_text_path = _shared_dir / "pruefung_gemeinsame_meldungen.csv"
_shared_comparison_path = _shared_dir / "vergleich_gemeinsame_meldungen.csv"
shared_review_texts.to_csv(_shared_text_path, index=False, sep=";", encoding="utf-8-sig")
shared_review_comparison.to_csv(_shared_comparison_path, index=False, sep=";", encoding="utf-8-sig")

print("Gemeinsame Stichprobe:", len(shared_review_texts), "Meldungen")
print(shared_review_counts.to_string(index=False))
print("Technische Kontrolle der Stichprobe: bestanden")
print("Gespeichert:", _shared_text_path.resolve())
print("Gespeichert:", _shared_comparison_path.resolve())


In [ ]:
# Schritt 11h Normalisierung und Vokabularfilter getrennt vergleichen
# Die Hauptanalyse bleibt unverändert, die 100 Meldungen aus Schritt 11g bleiben festgelegt.

from pathlib import Path
from copy import deepcopy
import hashlib
import json
import re
import warnings
import zipfile

import numpy as np
import pandas as pd
import sklearn
from joblib import hash as joblib_hash
from sklearn.base import clone


# Exakte Formen, keine Regeln für einzelne Melde-IDs oder Verwaltungskategorien.
# Die vorhandenen Zuordnungen werden ergänzt, eng verwandte Sachbegriffe zusammengeführt.
NORMALIZATION_GROUPS = {
    "müll": (
        "müll", "mülls", "abfall", "abfälle", "abfalls", "abfällen",
        "vermüllung", "vermüllt", "müllentsorgung", "abfallentsorgung",
        "müllablagerung", "müllablagerungen", "hausmüll", "restmüll",
    ),
    "sperrmüll": ("sperrmüll", "sperrmülls", "sperrmüllentsorgung", "spermuell"),
    "müllsack": ("müllsack", "müllsäcke", "müllsäcken"),
    "mülleimer": ("mülleimer", "mülleimern", "abfalleimer", "abfalleimern"),
    "sofa": ("sofa", "sofas", "sofateil", "sofateile", "sofateilen", "couch", "couches"),
    "fahrrad": (
        "fahrrad", "fahrrades", "fahrrads", "fahrräder", "fahrrädern",
        "fahrradleiche", "fahrradleichen", "radleiche", "radleichen",
        "schrottfahrrad", "schrottfahrräder", "schrottfahrrädern",
        "damenfahrrad", "herrenfahrrad", "kinderfahrrad", "kinderfahrräder",
        "leihfahrrad", "leihfahrräder",
    ),
    "beleuchtung": ("beleuchtung", "beleuchtungen"),
    "ampel": (
        "ampel", "ampeln", "ampelanlage", "ampelanlagen",
        "fußgängerampel", "fußgängerampeln", "fahrradampel", "fahrradampeln",
        "radfahrerampel", "radfahrerampeln",
    ),
    "schild": (
        "schild", "schilder", "schildern", "verkehrsschild", "verkehrsschilder",
        "verkehrsschildern", "straßenschild", "straßenschilder", "straßenschildern",
        "hinweisschild", "hinweisschilder", "hinweisschildern", "verkehrszeichen",
    ),
    "gehweg": (
        "gehweg", "gehwege", "gehwegen", "gehwegs", "gehweges",
        "fußweg", "fußwege", "fußwegen", "fußgängerweg", "fußgängerwege",
        "fußgängerwegen", "gehsteig", "gehsteige",
    ),
    "radweg": (
        "radweg", "radwege", "radwegen", "radwegs", "radweges",
        "fahrradweg", "fahrradwege", "fahrradwegen",
    ),
    "straßenschaden": ("straßenschaden", "straßenschäden"),
    "schlagloch": ("schlagloch", "schlaglöcher", "schlaglöchern"),
    "gulli": ("gulli", "gullis", "gully", "gullys", "gullydeckel", "gullideckel"),
    "ablauf": ("ablauf", "abläufe", "abläufen"),
    "baum": ("baum", "bäume", "bäumen", "baums", "baumes"),
    "glasflasche": ("glasflasche", "glasflaschen"),
}
# "Leuchten" kann ein Gegenstand oder ein Verb sein, etwa in "Ampeln leuchten".
# Die Form bleibt als eigener Begriff erhalten und wird nicht pauschal zu "beleuchtung".
NORMALIZATION_KEEP = {"leuchten"}
NORMALIZATION_PHRASES = (
    r"\bsehr\s+geehrte\s+damen\s+und\s+herren\b",
    r"\bmit\s+freundlichen\s+gr(?:ü|ue)(?:ß|ss)en\b",
)


def normalization_key(word: str) -> str:
    """Umlautschreibweisen nur für den Abgleich bekannter Wörter vereinheitlichen."""
    return word.lower().replace("ä", "ae").replace("ö", "oe").replace("ü", "ue").replace("ß", "ss")


def build_normalization_lookup(base: dict, groups: dict) -> dict:
    """Bekannte Schreibweisen zusammenführen und verkettete Zuordnungen auflösen."""
    lookup = {}
    for source, target in base.items():
        key = normalization_key(source)
        if key in lookup and normalization_key(lookup[key]) != normalization_key(target):
            raise ValueError(f"Widersprüchliche Begriffszuordnung für {source}.")
        lookup[key] = target
    for target, forms in groups.items():
        for form in forms:
            lookup[normalization_key(form)] = target
    resolved = {}
    for source in lookup:
        key, seen = source, set()
        while key in lookup:
            if key in seen:
                raise ValueError(f"Zyklische Begriffszuordnung für {source}.")
            seen.add(key)
            target = lookup[key]
            next_key = normalization_key(target)
            if next_key == key or next_key not in lookup:
                resolved[source] = target
                break
            key = next_key
    return resolved


def normalize_text_for_comparison(text: str, lookup: dict, stopwords: set, min_length: int) -> str:
    """Die begrenzten neuen Textregeln anwenden, ohne die Hauptfunktion zu ändern."""
    text = str(text).lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"<.*?>", " ", text)
    for expression in NORMALIZATION_PHRASES:
        text = re.sub(expression, " ", text)
    text = re.sub(r"[^a-zäöüß\s]", " ", text)
    tokens = []
    for original in text.split():
        token = lookup.get(normalization_key(original), original)
        if is_location_token(token):
            continue
        if normalization_key(token) not in stopwords and len(token) >= min_length:
            tokens.append(token)
    return " ".join(tokens)


def run_preprocessing_comparison():
    """Vier Varianten mit festen Modellen und derselben Stichprobe getrennt auswerten."""
    required = (
        "df_work", "df_model", "df_fit", "df_without_features", "model_mask",
        "X_count", "X_tfidf", "X_count_fit", "X_tfidf_fit",
        "count_vectorizer", "tfidf_vectorizer", "lda_model", "nmf_model",
        "lda_doc_topic", "nmf_doc_topic", "lda_assignments", "nmf_assignments",
        "shared_review_texts", "shared_review_comparison", "SELECTED_TOPIC_COUNTS",
        "CANONICAL_TERMS_FINAL", "STOPWORDS_FINAL", "clean_text", "is_location_token",
        "MIN_DF", "MIN_TOKEN_LENGTH", "MIN_TOKENS_CLEAN", "TEXT_COLUMN", "ID_COLUMN",
        "select_model_documents", "assign_topics", "normalize_nmf_for_assignment", "OUTPUT_DIR",
    )
    missing = [name for name in required if name not in globals()]
    if missing:
        raise RuntimeError("Zuerst die Zellen bis 11g ausführen. Es fehlen: " + ", ".join(missing))
    if SELECTED_TOPIC_COUNTS != {"LDA": 10, "NMF": 12} or MIN_DF != 5:
        raise ValueError("Der Vergleich benötigt die unveränderte Referenz mit LDA/10, NMF/12 und min_df=5.")
    templates = {"LDA": lda_model, "NMF": nmf_model}
    reference_assignments = {"LDA": lda_assignments, "NMF": nmf_assignments}
    reference_weights = {"LDA": lda_doc_topic, "NMF": nmf_doc_topic}
    for method, model in templates.items():
        if model.n_components != SELECTED_TOPIC_COUNTS[method]:
            raise ValueError("Die aktiven Modelle stimmen nicht mit Schritt 11f überein.")
    for vectorizer in (count_vectorizer, tfidf_vectorizer):
        if vectorizer.min_df != 5 or vectorizer.ngram_range != (1, 1):
            raise ValueError("Die Referenz muss unveränderte Unigramme mit min_df=5 verwenden.")
    if lda_model.doc_topic_prior is None or lda_model.topic_word_prior is None:
        raise ValueError("Die LDA-Priors müssen explizit festgelegt sein.")

    # Auch optionale bereits berechnete Ergebnistabellen in den Unverändert-Abgleich aufnehmen.
    protected_names = list(required) + [
        "topic_count_models", "topic_count_weights", "topic_count_assignments",
        "nmf_summary", "lda_summary", "decision_table", "decision_table_final", "decision_coverage",
    ]
    protected = {name: globals()[name] for name in protected_names
                 if name in globals() and not callable(globals()[name])}
    before = joblib_hash(protected)

    work = df_work.copy(deep=True)
    work[ID_COLUMN] = work[ID_COLUMN].astype("string").str.strip()
    if work[ID_COLUMN].isna().any() or work[ID_COLUMN].eq("").any() or work[ID_COLUMN].duplicated().any():
        raise ValueError("Die Melde-IDs des Arbeitsdatensatzes müssen vorhanden und eindeutig sein.")
    sample = shared_review_texts[["Prüffall", "ID", "Meldetext"]].copy(deep=True)
    sample["ID"] = sample["ID"].astype("string").str.strip()
    sample = sample.reset_index(drop=True)
    if len(sample) != 100 or sample["ID"].duplicated().any() or sample["Prüffall"].duplicated().any():
        raise ValueError("Erwartet werden die 100 eindeutigen Meldungen aus Schritt 11g.")
    if sample["ID"].isna().any() or not sample["ID"].isin(work[ID_COLUMN]).all():
        raise ValueError("Mindestens eine festgelegte Melde-ID fehlt im Arbeitsdatensatz.")
    original_texts = work.set_index(ID_COLUMN).loc[sample["ID"], TEXT_COLUMN].astype(str).to_numpy()
    if not np.array_equal(original_texts, sample["Meldetext"].astype(str).to_numpy()):
        raise ValueError("Die ursprünglichen Texte stimmen nicht mit Schritt 11g überein.")

    # Die Stichprobe nicht neu ziehen und bei späteren Ausführungen nicht stillschweigend ersetzen.
    output_dir = Path(OUTPUT_DIR) / "optimierung"
    output_dir.mkdir(parents=True, exist_ok=True)
    sample_path = output_dir / "stichprobe_festgelegt.csv"
    sample_csv = sample.to_csv(index=False, sep=";", lineterminator="\n")
    sample_sha = hashlib.sha256(sample_csv.encode("utf-8")).hexdigest()
    if sample_path.exists():
        frozen = pd.read_csv(sample_path, sep=";", dtype={"ID": "string"}, keep_default_na=False)
        frozen_csv = frozen.to_csv(index=False, sep=";", lineterminator="\n")
        if frozen_csv != sample_csv:
            raise ValueError("Die festgelegte Stichprobe weicht ab. Bestehende Prüfdatei nicht überschreiben.")

    lookup = build_normalization_lookup(dict(CANONICAL_TERMS_FINAL), NORMALIZATION_GROUPS)
    stopwords = {normalization_key(word) for word in STOPWORDS_FINAL}
    stopwords -= {normalization_key(word) for word in NORMALIZATION_KEEP}
    normalized = lambda text: normalize_text_for_comparison(text, lookup, stopwords, MIN_TOKEN_LENGTH)
    reference_texts = work[TEXT_COLUMN].apply(clean_text)
    normalized_texts = work[TEXT_COLUMN].apply(normalized)

    # Kleine Funktionstests prüfen die Regeln, nicht die inhaltliche Modellqualität.
    rule_checks = (
        (normalized("Müllentsorgung Vermuellung"), "müll müll"),
        (normalized("Fahrrädern Fahrraedern"), "fahrrad fahrrad"),
        (normalized("Straßenleuchte Strassenleuchte"), "beleuchtung beleuchtung"),
        (normalized("Die Ampeln leuchten nicht"), "ampel leuchten"),
        (normalized("Die Leuchten leuchten nicht"), "leuchten leuchten"),
        (normalized("Sehr geehrte Damen und Herren, Müll"), "müll"),
        (normalized("Sofas Glasflaschen"), "sofa glasflasche"),
        (normalized("Bitte danke"), ""),
    )
    if any(actual != expected for actual, expected in rule_checks):
        raise RuntimeError("Mindestens eine neue Textregel liefert ein unerwartetes Ergebnis.")
    restored = work.copy()
    restored["text_clean"] = reference_texts
    restored = restored.loc[reference_texts.str.split().str.len() >= MIN_TOKENS_CLEAN].reset_index(drop=True)
    main_ids = df_model[ID_COLUMN].astype("string").str.strip().reset_index(drop=True)
    if (not restored[ID_COLUMN].equals(main_ids)
            or not restored["text_clean"].reset_index(drop=True).equals(df_model["text_clean"].reset_index(drop=True))):
        raise ValueError("Die neu berechneten Referenztexte passen nicht zur aktiven Hauptanalyse.")

    variants = (
        ("Referenz", False, 5),
        ("Vokabularfilter", False, 2),
        ("Textnormalisierung", True, 5),
        ("Kombination", True, 2),
    )
    results, summary_rows, detail_rows, review_frames = {}, [], [], []
    for variant, use_normalization, min_df in variants:
        texts = normalized_texts if use_normalization else reference_texts
        docs = work.copy()
        docs["text_clean"] = texts
        docs["word_count_clean"] = texts.str.split().str.len()
        docs = docs.loc[docs["word_count_clean"] >= MIN_TOKENS_CLEAN].reset_index(drop=True)
        if docs.empty:
            raise ValueError(f"{variant}: Keine Texte nach Preprocessing vorhanden.")
        cv = clone(count_vectorizer).set_params(min_df=min_df)
        tv = clone(tfidf_vectorizer).set_params(min_df=min_df)
        count_matrix = cv.fit_transform(docs["text_clean"])
        tfidf_matrix = tv.fit_transform(docs["text_clean"])
        features = cv.get_feature_names_out()
        if not np.array_equal(features, tv.get_feature_names_out()):
            raise RuntimeError("Count und TF-IDF verwenden unterschiedliche Vokabulare.")
        mask, fit_docs, count_fit, tfidf_fit, without = select_model_documents(docs, count_matrix, tfidf_matrix)
        if variant == "Referenz":
            if (count_matrix.shape != X_count.shape or (count_matrix != X_count).nnz
                    or not np.array_equal(features, count_vectorizer.get_feature_names_out())):
                raise RuntimeError("Die Referenz reproduziert die Count-Matrix nicht.")
            delta = (tfidf_matrix - X_tfidf).tocsr()
            if delta.nnz and np.max(np.abs(delta.data)) > 1e-12:
                raise RuntimeError("Die Referenz reproduziert die TF-IDF-Matrix nicht.")
        result = {
            "documents": docs, "fit_documents": fit_docs, "mask": mask,
            "count_matrix": count_matrix, "tfidf_matrix": tfidf_matrix,
            "count_fit": count_fit, "tfidf_fit": tfidf_fit,
            "count_vectorizer": cv, "tfidf_vectorizer": tv,
            "models": {}, "weights": {}, "assignments": {},
        }
        all_clean = pd.Series(texts.to_numpy(), index=work[ID_COLUMN]).reindex(sample["ID"]).to_numpy()
        vocabulary = set(features)
        for method, matrix in (("LDA", count_fit), ("NMF", tfidf_fit)):
            print(f"{variant} | {method} mit {SELECTED_TOPIC_COUNTS[method]} Topics", flush=True)
            recorded = []
            if variant == "Referenz":
                model = deepcopy(templates[method])
                coefficients = np.asarray(reference_weights[method]).copy()
                assignments = reference_assignments[method].copy(deep=True)
                expected = assign_topics(coefficients, mask, docs.index)
                if not assignments.equals(expected):
                    raise RuntimeError("Die Referenzzuordnungen passen nicht zu den aktiven Koeffizienten.")
            else:
                model = clone(templates[method])
                with warnings.catch_warnings(record=True) as recorded:
                    warnings.simplefilter("always")
                    coefficients = model.fit_transform(matrix)
                if method == "NMF":
                    coefficients, _ = normalize_nmf_for_assignment(coefficients, model.components_)
                assignments = assign_topics(coefficients, mask, docs.index)
            if assignments.loc[~mask, "Topic"].notna().any():
                raise RuntimeError("Eine Meldung ohne Merkmale hat eine Topic-Zuordnung.")
            k = model.n_components
            if model.components_.shape != (k, len(features)) or not np.isfinite(model.components_).all():
                raise RuntimeError("Ungültige Topic-Vektoren im Vergleich.")
            counts = assignments["Topic"].value_counts().reindex(range(1, k + 1), fill_value=0)
            top_terms = {
                topic + 1: ", ".join(features[np.argsort(-model.components_[topic], kind="stable")[:10]])
                for topic in range(k)
            }
            warning_messages = [str(item.message) for item in recorded]
            for message in warning_messages:
                print("  Warnung:", message)
            by_id = assignments.copy()
            by_id.index = docs[ID_COLUMN]
            chosen = by_id.reindex(sample["ID"]).reset_index(drop=True)
            absent = ~sample["ID"].isin(docs[ID_COLUMN])
            chosen.loc[absent, "Status"] = "Durch Textfilter ausgeschlossen"
            if chosen["Status"].isna().any():
                raise RuntimeError("Für mindestens eine festgelegte Meldung fehlt der Status.")
            review = sample.copy()
            review.insert(0, "Variante", variant)
            review.insert(1, "Verfahren", method)
            review["Anzahl Topics"] = k
            review["min_df"] = min_df
            review["Text nach Preprocessing"] = all_clean
            review["Begriffe im Vokabular"] = [
                ", ".join(dict.fromkeys(word for word in text.split() if word in vocabulary))
                for text in all_clean
            ]
            review["Topic"] = chosen["Topic"].astype("Int64")
            review["Status"] = chosen["Status"]
            review["Top-Begriffe"] = chosen["Topic"].map(top_terms).fillna("")
            if variant == "Referenz":
                prior = shared_review_comparison.set_index("ID").copy()
                prior.index = prior.index.astype("string")
                prior = prior.loc[sample["ID"]].reset_index(drop=True)
                if (not np.array_equal(review["Topic"].fillna(-1).to_numpy(), prior[f"{method} Topic"].fillna(-1).to_numpy())
                        or not review["Status"].equals(prior[f"{method} Status"])
                        or not review["Top-Begriffe"].equals(prior[f"{method} Top-Begriffe"])):
                    raise RuntimeError("Die Referenz weicht vom Vergleich der 100 Meldungen ab.")
            summary_rows.append({
                "Variante": variant, "Verfahren": method, "Anzahl Topics": k, "min_df": min_df,
                "Neue Textregeln": use_normalization, "Neu trainiert": variant != "Referenz",
                "Meldungen im Arbeitsdatensatz": len(work), "Texte verändert": int(texts.ne(reference_texts).sum()),
                "Meldungen nach Text Preprocessing": len(docs), "Durch Textfilter ausgeschlossen": len(work) - len(docs),
                "Merkmale": len(features), "Ohne Merkmale": len(without), "Für das Modelltraining": len(fit_docs),
                "Topic zugeordnet": int(assignments["Topic"].notna().sum()),
                "Ohne positive Topic-Koeffizienten": int(assignments.loc[mask, "Topic"].isna().sum()),
                "Topics ohne Zuordnung": int(counts.eq(0).sum()), "Stichprobe": len(sample),
                "Stichprobe mit Zuordnung": int(review["Topic"].notna().sum()),
                "Stichprobe ohne Zuordnung": int(review["Topic"].isna().sum()),
                "Iterationen": int(model.n_iter_), "Warnungen neuer Lauf": len(recorded),
                "Hinweise": " | ".join(warning_messages),
            })
            for topic in range(1, k + 1):
                detail_rows.append({
                    "Variante": variant, "Verfahren": method, "Anzahl Topics": k,
                    "Topic": topic, "Anzahl Meldungen": int(counts.loc[topic]),
                    "Anteil Prozent": 100 * int(counts.loc[topic]) / len(docs),
                    "Bezugsmenge Meldungen": len(docs), "Top-Begriffe": top_terms[topic],
                })
            result["models"][method] = model
            result["weights"][method] = coefficients
            result["assignments"][method] = assignments
            review_frames.append(review)
        results[variant] = result

    summary = pd.DataFrame(summary_rows)
    details = pd.DataFrame(detail_rows)
    review = pd.concat(review_frames, ignore_index=True)
    if len(review) != 800 or review.duplicated(["Variante", "Verfahren", "ID"]).any():
        raise RuntimeError("Erwartet werden acht Auswertungen derselben 100 Meldungen.")
    for _, part in review.groupby(["Variante", "Verfahren"], sort=False):
        if not part[["Prüffall", "ID", "Meldetext"]].reset_index(drop=True).equals(sample):
            raise RuntimeError("Die Stichprobe hat sich zwischen Varianten verändert.")
    if before != joblib_hash(protected):
        raise RuntimeError("Die Hauptanalyse wurde während des Vergleichs verändert.")

    settings = {
        "vergleich": "Vier Varianten mit LDA/10 und NMF/12, keine neue Auswahl der Topic-Anzahl",
        "referenz": "Vorhandene Modelle und Zuordnungen aus Schritt 11f, nicht neu trainiert",
        "stichprobe": {"anzahl": len(sample), "sha256_csv_utf8": sample_sha, "quelle": "Schritt 11g, nicht neu gezogen"},
        "df_work_sha256": hashlib.sha256(
            work[[ID_COLUMN, TEXT_COLUMN]].to_csv(index=False, lineterminator="\n").encode("utf-8")
        ).hexdigest(),
        "varianten": [{"name": v, "neue_textregeln": n, "min_df": m} for v, n, m in variants],
        "wortformen": NORMALIZATION_GROUPS, "beibehalten": sorted(NORMALIZATION_KEEP),
        "entfernte_floskeln": list(NORMALIZATION_PHRASES), "aufgeloeste_zuordnungen": lookup,
        "bisherige_zuordnungen": dict(CANONICAL_TERMS_FINAL), "bisherige_stopwoerter": sorted(STOPWORDS_FINAL),
        "min_tokens_clean": MIN_TOKENS_CLEAN, "min_token_length": MIN_TOKEN_LENGTH,
        "vectorizer_count": count_vectorizer.get_params(), "vectorizer_tfidf": tfidf_vectorizer.get_params(),
        "modelle": {method: model.get_params() for method, model in templates.items()},
        "zuordnung_nmf": "W-Spalten mit H-Zeilennorm multipliziert, H-Zeilen L2-normiert",
        "coherence": "In dieser Runde nicht berechnet, erst für eine inhaltlich aussichtsreiche Variante",
        "bewertung": "Nicht automatisch durchgeführt. Kriterien und bisherige Einzelbewertungen bleiben unverändert.",
        "versionen": {"numpy": np.__version__, "pandas": pd.__version__, "sklearn": sklearn.__version__},
    }
    paths = [sample_path]
    if not sample_path.exists():
        sample_path.write_text(sample_csv, encoding="utf-8-sig")
    for name, frame in (("vergleich_textverarbeitung.csv", summary),
                        ("topics_textverarbeitung.csv", details),
                        ("pruefung_textverarbeitung.csv", review)):
        path = output_dir / name
        frame.to_csv(path, index=False, sep=";", encoding="utf-8-sig", lineterminator="\n")
        paths.append(path)
    settings_path = output_dir / "einstellungen_textverarbeitung.json"
    settings_path.write_text(json.dumps(settings, ensure_ascii=False, indent=2, default=str) + "\n", encoding="utf-8")
    paths.append(settings_path)
    archive = output_dir / "vergleich_textverarbeitung.zip"
    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
        for path in paths:
            bundle.write(path, arcname=path.name)

    display(summary[["Variante", "Verfahren", "min_df", "Meldungen nach Text Preprocessing",
                     "Merkmale", "Ohne Merkmale", "Für das Modelltraining", "Stichprobe ohne Zuordnung"]])
    print("Festgelegte Stichprobe:", len(sample), "Meldungen")
    print("Modelle im Vergleich:", len(summary), "(zwei Referenzmodelle, sechs neue Trainingsläufe)")
    print("Zeilen für die inhaltliche Prüfung:", len(review))
    print("Technische Kontrolle des Variantenvergleichs: bestanden")
    print("Die Hauptanalyse und ihre bisherigen Zuordnungen sind unverändert.")
    print("Gespeichert:", archive.resolve())
    return results, summary, details, review, settings


(
    optimization_results, optimization_summary, optimization_details,
    optimization_review, optimization_settings,
) = run_preprocessing_comparison()


In [ ]:
# Schritt 11i Vergleich der Kategorien und Dominanz der vorhandenen Zuordnungen prüfen
# Nur Diagnose als Diagnose vorgesehen

from pathlib import Path
import hashlib
import json
import zipfile

import numpy as np
import pandas as pd
import sklearn
from joblib import hash as joblib_hash
from sklearn.metrics import (
    normalized_mutual_info_score, adjusted_mutual_info_score, adjusted_rand_score,
)

# Vergleichsgrenzen für die Diagnose
DIAG_THRESHOLDS = (0.4, 0.5, 0.6)
DIAG_CATEGORIES = ("category_parent", "category_child")
DIAG_VARIANTS = ("Referenz", "Vokabularfilter", "Textnormalisierung", "Kombination")


def diagnostic_fingerprint(value) -> str:
    """Inhalte vergleichen, ohne interne pandas-Caches als Datenänderung zu behandeln."""
    if isinstance(value, pd.DataFrame):
        value = (list(value.columns), list(value.dtypes.astype(str)), value.index.names,
                 pd.util.hash_pandas_object(value, index=True).to_numpy())
    elif isinstance(value, pd.Series):
        value = (value.name, str(value.dtype), value.index.names,
                 pd.util.hash_pandas_object(value, index=True).to_numpy())
    elif isinstance(value, dict):
        value = {key: diagnostic_fingerprint(item) for key, item in value.items()}
    elif isinstance(value, (tuple, list)):
        value = [diagnostic_fingerprint(item) for item in value]
    return joblib_hash(value)


def diagnostic_ids(values: pd.Series) -> pd.Series:
    ids = values.astype("string").str.strip()
    if ids.isna().any() or ids.eq("").any() or ids.duplicated().any():
        raise ValueError("Für die Diagnose werden vorhandene, eindeutige Melde-IDs benötigt.")
    return ids


def topic_dominance(coefficients: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Anteil des stärksten Werts berechnen, ohne daraus eine Trefferwahrscheinlichkeit abzuleiten."""
    w = np.asarray(coefficients, dtype=float)
    if w.ndim != 2 or w.shape[1] < 2 or not np.isfinite(w).all() or (w < 0).any():
        raise ValueError("Die Dokument-Topic-Matrix muss endlich, nicht negativ und zweidimensional sein.")
    highest = w.max(axis=1)
    positive = highest > 0
    dominance = np.full(len(w), np.nan)
    # Diese rechnerisch gleichwertige Form vermeidet einen Überlauf bei großen Summen an Zeilen.
    dominance[positive] = 1.0 / (w[positive] / highest[positive, None]).sum(axis=1)
    ties = positive & ((w == highest[:, None]).sum(axis=1) > 1)
    return dominance, ties


def category_statistics(frame: pd.DataFrame, chosen: pd.Series, column: str) -> tuple[dict, pd.DataFrame]:
    """Kategorien mit den Topics einer ausdrücklich ausgewählten Menge vergleichen."""
    chosen = chosen.fillna(False)
    selected = frame.loc[chosen, ["Topic", column]].copy()
    missing = int(selected[column].isna().sum())
    valid = selected.dropna(subset=[column])
    if valid["Topic"].isna().any():
        raise ValueError("Die Kategorienkennzahlen benötigen vorhandene Topic-Zuordnungen.")
    table = pd.crosstab(valid["Topic"].astype(int), valid[column])
    table.index.name, table.columns.name = "Topic", "Kategorie"
    n = len(valid)
    scores = {name: np.nan for name in ("NMI", "AMI", "ARI")}
    if n >= 2:
        labels, topics = valid[column].astype(str).to_numpy(), valid["Topic"].astype(int).to_numpy()
        scores = {
            "NMI": normalized_mutual_info_score(labels, topics, average_method="arithmetic"),
            "AMI": adjusted_mutual_info_score(labels, topics, average_method="arithmetic"),
            "ARI": adjusted_rand_score(labels, topics),
        }
    scores.update({
        "Ausgewählte Meldungen": len(selected), "Ohne Kategorie": missing,
        "Bewertete Meldungen": n, "Belegte Topics": valid["Topic"].nunique(),
        "Vorhandene Kategorien": valid[column].nunique(),
        "Reinheit": float(table.max(axis=1).sum() / n) if n else np.nan,
    })
    return scores, table


def diagnostic_frames(results: dict, source: pd.DataFrame, id_column: str, text_column: str) -> tuple[dict, dict]:
    """Alle Varianten über Melde-IDs verbinden und die Zuordnung aus den gespeicherten Werten prüfen."""
    required = {id_column, text_column, *DIAG_CATEGORIES}
    if source.empty or not source.columns.is_unique or not required.issubset(source.columns):
        raise ValueError("Im Arbeitsdatensatz fehlen eindeutige Spalten, Kategorien oder Meldungen.")
    if set(results) != set(DIAG_VARIANTS):
        raise ValueError("Erwartet werden die vier unveränderten Varianten aus Schritt 11h.")
    source = source.copy()
    source[id_column] = diagnostic_ids(source[id_column])
    original = source.set_index(id_column)
    for column in DIAG_CATEGORIES:
        original[column] = original[column].astype("string").str.strip().replace("", pd.NA)
    frames, metadata = {}, {}
    for variant in DIAG_VARIANTS:
        result = results[variant]
        docs, fit_docs = result["documents"], result["fit_documents"]
        ids = diagnostic_ids(docs[id_column])
        mask = np.asarray(result["mask"])
        if mask.dtype != bool or mask.shape != (len(docs),) or not docs.index.is_unique:
            raise ValueError(f"{variant}: Die Auswahl der Trainingsmeldungen ist ungültig.")
        if not docs.index[mask].equals(fit_docs.index):
            raise ValueError(f"{variant}: Die Reihenfolge der Meldungen stimmt nicht.")
        if not docs.loc[mask, [id_column, "text_clean"]].equals(fit_docs[[id_column, "text_clean"]]):
            raise ValueError(f"{variant}: Die Trainingsmeldungen gehören nicht zu diesem Datenstand.")
        if not ids.isin(original.index).all():
            raise ValueError(f"{variant}: Eine Melde-ID fehlt im Arbeitsdatensatz.")
        if not np.array_equal(docs[text_column].astype(str), original.loc[ids, text_column].astype(str)):
            raise ValueError(f"{variant}: Die originalen Meldungen wurden seit Schritt 11h verändert.")
        for column in DIAG_CATEGORIES:
            values = docs[column].astype("string").str.strip().replace("", pd.NA).reset_index(drop=True)
            if not values.equals(original.loc[ids, column].reset_index(drop=True)):
                raise ValueError(f"{variant}: Die Kategorien stimmen nicht mit dem Arbeitsdatensatz überein.")
        # Sparse-Vergleiche können intern sortieren, deshalb nur eine Kopie verwenden.
        count_matrix = result["count_matrix"].copy()
        if count_matrix.shape[0] != len(docs):
            raise ValueError(f"{variant}: Merkmalsmatrix und Meldungen passen nicht zusammen.")
        feature_counts = np.asarray((count_matrix != 0).sum(axis=1)).ravel()
        if not np.array_equal(feature_counts > 0, mask):
            raise ValueError(f"{variant}: Merkmalsmatrix und Trainingsauswahl widersprechen sich.")
        for method in ("LDA", "NMF"):
            model, assignments = result["models"][method], result["assignments"][method]
            w = np.asarray(result["weights"][method], dtype=float)
            if w.shape != (len(fit_docs), model.n_components) or not assignments.index.equals(docs.index):
                raise ValueError(f"{variant}/{method}: Koeffizienten und Zuordnungen passen nicht zusammen.")
            # Die NMF-Werte aus 11h sind bereits auf L2-normierte Topic-Vektoren abgestimmt.
            dominance, ties = topic_dominance(w)
            positive = np.isfinite(dominance)
            expected = np.full(len(docs), -1, dtype=int)
            expected[np.flatnonzero(mask)[positive]] = w[positive].argmax(axis=1) + 1
            actual = assignments["Topic"].to_numpy(dtype=int, na_value=-1)
            if not np.array_equal(expected, actual):
                raise ValueError(f"{variant}/{method}: Topic-Zuordnungen passen nicht zu den Koeffizienten.")
            expected_status = np.full(len(docs), "Keine Merkmale im Vokabular", dtype=object)
            expected_status[mask] = "Keine positiven Topic-Gewichte"
            expected_status[expected > 0] = "Topic zugeordnet"
            if not np.array_equal(expected_status, assignments["Status"].astype(str)):
                raise ValueError(f"{variant}/{method}: Topic und Status widersprechen sich.")
            frame = original[list(DIAG_CATEGORIES)].copy()
            frame.index.name = "ID"
            frame["Nach Text Preprocessing"] = frame.index.isin(ids)
            frame["Topic"] = pd.Series(assignments["Topic"].array, index=ids).reindex(frame.index)
            status = pd.Series(assignments["Status"].to_numpy(), index=ids)
            frame["Status"] = status.reindex(frame.index).fillna("Durch Textfilter ausgeschlossen")
            frame["Dominanz"] = pd.Series(dominance, index=ids.loc[mask]).reindex(frame.index)
            frame["Mehrere Höchstwerte"] = pd.Series(ties, index=ids.loc[mask], dtype="boolean").reindex(frame.index).fillna(False)
            frame["Merkmale in der Meldung"] = pd.Series(feature_counts, index=ids, dtype="Int64").reindex(frame.index)
            for threshold in DIAG_THRESHOLDS:
                frame[f"Dominanz >= {threshold:.1f}"] = frame["Topic"].notna() & frame["Dominanz"].ge(threshold)
            vectorizer = result["tfidf_vectorizer" if method == "NMF" else "count_vectorizer"]
            vocabulary = vectorizer.get_feature_names_out()
            h = np.asarray(model.components_)
            if h.shape != (model.n_components, len(vocabulary)) or not np.isfinite(h).all() or (h < 0).any():
                raise ValueError(f"{variant}/{method}: Topic-Vektoren und Vokabular passen nicht zusammen.")
            top = {i + 1: ", ".join(vocabulary[np.argsort(-row, kind="stable")[:10]]) for i, row in enumerate(h)}
            key = (variant, method)
            frames[key] = frame
            metadata[key] = {"Variante": variant, "Verfahren": method, "Anzahl Topics": model.n_components,
                             "min_df": vectorizer.min_df, "Merkmale": len(vocabulary), "top_terms": top}
    return frames, metadata


def run_category_diagnostics(results: dict, source: pd.DataFrame, sample: pd.DataFrame,
                             id_column: str, text_column: str, output_dir: Path,
                             previous_settings: dict) -> dict:
    """Kategorien, Dominanz und Abdeckung auswerten. Die bisherigen Modelle bleiben erhalten."""
    before = diagnostic_fingerprint((results, source, sample, previous_settings))
    frames, metadata = diagnostic_frames(results, source, id_column, text_column)
    first = next(iter(frames.values()))
    common = pd.Series(True, index=first.index)
    for frame in frames.values():
        common &= frame["Topic"].notna()
    if not common.any():
        raise ValueError("Es gibt keine in allen acht Modellvarianten zugeordneten Meldungen.")
    review_ids = diagnostic_ids(sample["ID"])
    if not review_ids.isin(first.index).all() or len(review_ids) != 100:
        raise ValueError("Die festgelegte Stichprobe aus 11g muss 100 bekannte IDs enthalten.")
    original = source.assign(**{id_column: diagnostic_ids(source[id_column])}).set_index(id_column)
    if not np.array_equal(sample["Meldetext"].astype(str), original.loc[review_ids, text_column].astype(str)):
        raise ValueError("Die Texte der festgelegten Stichprobe stimmen nicht mit dem Arbeitsdatensatz überein.")
    category_rows, dominance_rows, topic_rows, cross_rows = [], [], [], []
    all_rows, sample_rows = [], []
    for key, frame in frames.items():
        meta = {name: value for name, value in metadata[key].items() if name != "top_terms"}
        assigned = frame["Topic"].notna()
        after = int(frame["Nach Text Preprocessing"].sum())
        coverage = {
            "Meldungen im Arbeitsdatensatz": len(frame), "Nach Text Preprocessing": after,
            "Durch Textfilter ausgeschlossen": int((~frame["Nach Text Preprocessing"]).sum()),
            "Mit Topic-Zuordnung": int(assigned.sum()),
            "Ohne Merkmale": int(frame["Status"].eq("Keine Merkmale im Vokabular").sum()),
            "Ohne positive Koeffizienten": int(frame["Status"].eq("Keine positiven Topic-Gewichte").sum()),
        }
        for basis, chosen in (("Jeweils zugeordnet", assigned), ("Gemeinsam zugeordnet", common)):
            for column in DIAG_CATEGORIES:
                scores, table = category_statistics(frame, chosen, column)
                category_rows.append({**meta, "Kategorieebene": column, "Datenbasis": basis, **coverage, **scores})
                if basis == "Jeweils zugeordnet":
                    for topic, row in table.iterrows():
                        for category, count in row.items():
                            cross_rows.append({**meta, "Kategorieebene": column, "Auswahl": "Ohne Dominanzfilter",
                                               "Topic": topic, "Kategorie": category, "Anzahl": int(count)})
        for threshold in DIAG_THRESHOLDS:
            high = frame[f"Dominanz >= {threshold:.1f}"]
            low = assigned & ~high
            for part, chosen in (("Mindestens Schwelle", high), ("Unter Schwelle", low)):
                for column in DIAG_CATEGORIES:
                    scores, table = category_statistics(frame, chosen, column)
                    dominance_rows.append({**meta, "Kategorieebene": column, "Schwelle": threshold,
                        "Teilmenge": part, **coverage, **scores,
                        "Anteil an Zuordnungen in Prozent": 100 * chosen.sum() / assigned.sum() if assigned.any() else np.nan,
                        "Anteil nach Preprocessing in Prozent": 100 * chosen.sum() / after if after else np.nan,
                        "Anteil am Arbeitsdatensatz in Prozent": 100 * chosen.sum() / len(frame)})
                    if threshold == 0.5:
                        for topic, row in table.iterrows():
                            for category, count in row.items():
                                cross_rows.append({**meta, "Kategorieebene": column,
                                    "Auswahl": "Dominanz >= 0.5" if part == "Mindestens Schwelle" else "Dominanz < 0.5",
                                    "Topic": topic, "Kategorie": category, "Anzahl": int(count)})
            for topic in range(1, meta["Anzahl Topics"] + 1):
                group = frame["Topic"].eq(topic).fillna(False)
                n, kept = int(group.sum()), int((group & high).sum())
                topic_rows.append({**meta, "Topic": topic, "Top-Begriffe": metadata[key]["top_terms"][topic],
                    "Schwelle": threshold, "Zugeordnet": n, "Mindestens Schwelle": kept, "Unter Schwelle": n - kept,
                    "Anteil im Topic in Prozent": 100 * kept / n if n else np.nan,
                    "Mehrere Höchstwerte": int((group & frame["Mehrere Höchstwerte"]).sum())})
        part = frame.copy()
        part["Gemeinsam zugeordnet"] = common
        for name, value in reversed(list(meta.items())):
            part.insert(0, name, value)
        all_rows.append(part.reset_index())
        review = part.loc[review_ids].copy().reset_index()
        review.insert(0, "Prüffall", sample["Prüffall"].to_numpy())
        review["Top-Begriffe"] = review["Topic"].map(metadata[key]["top_terms"]).fillna("")
        sample_rows.append(review)
    tables = {
        "vergleich_kategorien": pd.DataFrame(category_rows),
        "vergleich_dominanz": pd.DataFrame(dominance_rows),
        "dominanz_je_topic": pd.DataFrame(topic_rows),
        "kreuztabellen_kategorien": pd.DataFrame(cross_rows),
        "dominanz_meldungen": pd.concat(all_rows, ignore_index=True),
        "dominanz_stichprobe": pd.concat(sample_rows, ignore_index=True),
    }
    if before != diagnostic_fingerprint((results, source, sample, previous_settings)):
        raise RuntimeError("Eingaben oder Modelle wurden während der Diagnose verändert.")
    out = Path(output_dir) / "optimierung" / "kategorien_dominanz"
    out.mkdir(parents=True, exist_ok=True)
    paths = []
    for name, table in tables.items():
        path = out / f"{name}.csv"
        table.to_csv(path, index=False, sep=";", encoding="utf-8-sig", lineterminator="\n")
        paths.append(path)
    settings = {
        "zweck": "Diagnose vorhandener Ergebnisse aus 11h, keine geänderte Zähl- oder Freigaberegel",
        "schwellen": list(DIAG_THRESHOLDS), "vergleichsoperator": ">=", "rundung_vor_vergleich": False,
        "dominanz": "max(w) / sum(w), NMF bereits L2-skaliert aus 11h; keine kalibrierte Trefferwahrscheinlichkeit",
        "gleichstaende": "Exakt gleiche Höchstwerte werden ausgewiesen, die Diagnose schließt sie nicht zusätzlich aus.",
        "kategorien": list(DIAG_CATEGORIES), "fehlende_kategorien": "Aus Kennzahlen ausgeschlossen und gezählt",
        "gemeinsame_basis": "Schnittmenge zugeordneter IDs aller vier Varianten und beider Verfahren, ohne Dominanzfilter",
        "gemeinsam_zugeordnet": int(common.sum()),
        "dominanz_teilmengen": "Je Modell getrennt. Kennzahlen gelten für unterschiedliche Mengen, nicht direkt als Qualitätssteigerung.",
        "reinheit": "Summe der jeweils häufigsten Verwaltungskategorie pro Topic geteilt durch die bewertete Anzahl",
        "nmi_ami_normalisierung": "arithmetic", "stichprobe": "Unveränderte 100 Fälle aus 11g, keine neue Bewertung",
        "eingaben_joblib_hash": before, "versionen": {"numpy": np.__version__, "pandas": pd.__version__, "sklearn": sklearn.__version__},
        "einstellungen_11h": previous_settings,
        "sha256_ausgaben": {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in paths},
    }
    path = out / "einstellungen_kategorien_dominanz.json"
    path.write_text(json.dumps(settings, ensure_ascii=False, indent=2, default=str) + "\n", encoding="utf-8")
    paths.append(path)
    archive = out / "vergleich_kategorien_dominanz.zip"
    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
        for path in paths:
            bundle.write(path, arcname=path.name)
    print("Kategorienvergleich ohne Dominanzfilter, gemeinsame Melde-IDs")
    overview = tables["vergleich_kategorien"]
    display(overview.loc[overview["Datenbasis"].eq("Gemeinsam zugeordnet") & overview["Kategorieebene"].eq("category_parent"),
        ["Variante", "Verfahren", "Bewertete Meldungen", "NMI", "AMI", "ARI", "Reinheit"]].round(4))
    print("Dominanz mindestens 0,5, jeweils eigene Auswahl")
    overview = tables["vergleich_dominanz"]
    display(overview.loc[overview["Schwelle"].eq(0.5) & overview["Teilmenge"].eq("Mindestens Schwelle")
        & overview["Kategorieebene"].eq("category_parent"),
        ["Variante", "Verfahren", "Mit Topic-Zuordnung", "Ausgewählte Meldungen",
         "Anteil nach Preprocessing in Prozent", "NMI", "AMI", "ARI", "Reinheit"]].round(4))
    print("Ausgewertete Modellvarianten:", len(frames))
    print("Gemeinsam zugeordnete Meldungen:", int(common.sum()))
    print("Schwellen für die Diagnose:", ", ".join(str(t) for t in DIAG_THRESHOLDS))
    print("Zeilen zur festgelegten Stichprobe:", len(tables["dominanz_stichprobe"]))
    print("Technische Kontrolle der Kategorien- und Dominanzdiagnostik: bestanden")
    print("Die Hauptanalyse und ihre bisherigen Zuordnungen sind unverändert.")
    print("Gespeichert:", archive.resolve())
    return {**tables, "einstellungen": settings, "zip_path": archive}


# Die Koeffizienten liegen im Kernel, sie können nicht aus der bisherigen Ergebnis-ZIP geladen werden.
_diag_required = ("optimization_results", "optimization_settings", "df_work", "shared_review_texts", "ID_COLUMN", "TEXT_COLUMN", "OUTPUT_DIR")
_diag_missing = [name for name in _diag_required if name not in globals()]
if _diag_missing:
    raise RuntimeError("Zuerst die Zellen bis 11h ausführen. Es fehlen: " + ", ".join(_diag_missing))
category_diagnostics = run_category_diagnostics(
    optimization_results, df_work, shared_review_texts, ID_COLUMN, TEXT_COLUMN,
    OUTPUT_DIR, optimization_settings,
)


In [ ]:
# Schritt 11j Die Anzahl der NMF-Topics für die beiden Varianten der Normalisierung vergleichen
# Textregeln und Ergebniszählung der Hauptanalyse bleiben unverändert.

from copy import deepcopy
from pathlib import Path
import hashlib
import json
import warnings
import zipfile

import numpy as np
import pandas as pd
import sklearn
import gensim
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
from sklearn.base import clone

NMF_VARIANTS_TO_COMPARE = ("Textnormalisierung", "Kombination")
NMF_TOPIC_COUNTS = (6, 8, 10, 12, 15)


def compare_normalized_nmf_counts(results, diagnostics, source, sample, previous_settings,
                                  output_dir, id_column, text_column):
    """Feste Textvarianten bei unterschiedlichen Topic-Anzahlen auswerten."""
    protected = (results, diagnostics, source, sample, previous_settings)
    before = diagnostic_fingerprint(protected)
    base_frames, _ = diagnostic_frames(results, source, id_column, text_column)
    if tuple(diagnostics["einstellungen"]["schwellen"]) != tuple(DIAG_THRESHOLDS):
        raise ValueError("Die Diagnosegrenzen stimmen nicht mit Schritt 11i überein.")
    if diagnostic_fingerprint(diagnostics["einstellungen"]["einstellungen_11h"]) != diagnostic_fingerprint(previous_settings):
        raise ValueError("Schritt 11i und die Varianten aus 11h haben unterschiedliche Einstellungen.")
    if not NMF_TOPIC_COUNTS or len(set(NMF_TOPIC_COUNTS)) != len(NMF_TOPIC_COUNTS):
        raise ValueError("Die vorgegebenen Topic-Anzahlen müssen eindeutig sein.")
    if any(isinstance(k, bool) or not isinstance(k, int) or k < 2 for k in NMF_TOPIC_COUNTS):
        raise ValueError("Für jeden Versuch sind mindestens zwei Topics erforderlich.")

    originals = source.copy()
    originals[id_column] = diagnostic_ids(originals[id_column])
    originals = originals.set_index(id_column)
    original_categories = originals[list(DIAG_CATEGORIES)].astype("string").apply(
        lambda column: column.str.strip().replace("", pd.NA)
    )
    sample = sample.copy(deep=True)
    sample["ID"] = diagnostic_ids(sample["ID"])
    if len(sample) != 100 or not sample["ID"].isin(originals.index).all():
        raise ValueError("Erwartet werden die festgelegten 100 Meldungen aus Schritt 11g.")
    if not np.array_equal(sample["Meldetext"].astype(str), originals.loc[sample["ID"], text_column].astype(str)):
        raise ValueError("Die Meldetexte der festgelegten Stichprobe stimmen nicht überein.")

    # Der Kategorienvergleich verwendet zunächst dieselbe gemeinsame Menge wie 11i.
    common = pd.Series(True, index=originals.index)
    for frame in base_frames.values():
        common &= frame["Topic"].notna().reindex(common.index, fill_value=False)
    anchor_count = int(common.sum())
    if anchor_count != diagnostics["einstellungen"]["gemeinsam_zugeordnet"]:
        raise ValueError("Die gemeinsame Menge stimmt nicht mit Schritt 11i überein.")

    # Beide Varianten enthalten dieselben normalisierten Texte, aber verschiedene Vokabulare.
    reference = None
    for name, expected_min_df in zip(NMF_VARIANTS_TO_COMPARE, (5, 2)):
        data = results[name]
        vectorizer = data["tfidf_vectorizer"]
        if vectorizer.min_df != expected_min_df or vectorizer.ngram_range != (1, 1):
            raise ValueError(f"{name}: Erwartet werden Unigramme mit min_df={expected_min_df}.")
        docs = data["documents"][[id_column, "text_clean"]].copy()
        docs[id_column] = diagnostic_ids(docs[id_column])
        if reference is None:
            reference = docs
        elif not reference.equals(docs):
            raise ValueError("Die beiden Varianten verwenden nicht dieselben normalisierten Meldungen.")
        matrix = data["tfidf_fit"]
        if max(NMF_TOPIC_COUNTS) > min(matrix.shape):
            raise ValueError("Für diese NMF-Initialisierung sind zu wenige Meldungen oder Merkmale vorhanden.")
        if not np.isfinite(matrix.data).all() or (matrix.data < 0).any():
            raise ValueError("Die TF-IDF-Matrix enthält ungültige Werte.")

    reference_texts = reference["text_clean"].str.split().tolist()
    if any(not tokens for tokens in reference_texts):
        raise ValueError("Die Texte für c_v dürfen nicht leer sein.")
    dictionary = Dictionary(reference_texts)
    top_n, window_size = 10, 110
    comparison_models, model_rows, topic_rows = {}, [], []
    total = len(NMF_VARIANTS_TO_COMPARE) * len(NMF_TOPIC_COUNTS)
    print("Meldungen für c_v:", len(reference_texts))
    print("Gemeinsame Meldungen aus Schritt 11i:", anchor_count)

    for name in NMF_VARIANTS_TO_COMPARE:
        data = results[name]
        docs, fit_docs = data["documents"], data["fit_documents"]
        ids = diagnostic_ids(docs[id_column])
        mask = np.asarray(data["mask"])
        vectorizer = data["tfidf_vectorizer"]
        features = vectorizer.get_feature_names_out()
        if len(features) < top_n or not set(features).issubset(dictionary.token2id):
            raise ValueError("Das Vokabular passt nicht zu den Texten für c_v.")
        template = data["models"]["NMF"]
        if template.n_components != 12 or template.random_state != 42:
            raise ValueError("Die Kandidaten müssen dem NMF-Stand mit zwölf Topics und Seed 42 aus 11h entsprechen.")
        for k in NMF_TOPIC_COUNTS:
            reused = k == template.n_components
            action = "Modell aus 11h" if reused else "neues Training"
            print(f"[{len(model_rows) + 1}/{total}] {name}, {k} Topics ({action})", flush=True)
            recorded = []
            if reused:
                model = deepcopy(template)
                coefficients = np.asarray(data["weights"]["NMF"], dtype=float).copy()
            else:
                model = clone(template).set_params(n_components=k)
                with warnings.catch_warnings(record=True) as recorded:
                    warnings.simplefilter("always")
                    raw = model.fit_transform(data["tfidf_fit"].copy())
                coefficients, _ = normalize_nmf_for_assignment(raw, model.components_)
            for warning in recorded:
                print("  Warnung:", str(warning.message))
            assignments = assign_topics(coefficients, mask, docs.index)
            if reused and not assignments.equals(data["assignments"]["NMF"]):
                raise RuntimeError("Die übernommenen Zuordnungen weichen von Schritt 11h ab.")
            h = np.asarray(model.components_)
            if h.shape != (k, len(features)) or not np.isfinite(h).all() or (h < 0).any():
                raise RuntimeError("Die NMF-Komponenten passen nicht zum Vokabular.")
            if ((h > 0).sum(axis=1) < top_n).any():
                raise RuntimeError("Ein Topic enthält zu wenige positive Begriffe für c_v.")
            topics = [features[indices].tolist() for indices in np.argsort(-h, axis=1, kind="stable")[:, :top_n]]
            coherence = CoherenceModel(
                topics=topics, texts=reference_texts, dictionary=dictionary,
                coherence="c_v", topn=top_n, window_size=window_size, processes=1,
            )
            per_topic = np.asarray(coherence.get_coherence_per_topic(), dtype=float)
            if per_topic.shape != (k,) or not np.isfinite(per_topic).all():
                raise RuntimeError("Die Kohärenzwerte sind nicht vollständig berechenbar.")

            dominance, ties = topic_dominance(coefficients)
            frame = original_categories.copy()
            frame.index.name = "ID"
            frame["Nach Text Preprocessing"] = frame.index.isin(ids)
            frame["Topic"] = pd.Series(assignments["Topic"].array, index=ids).reindex(frame.index)
            frame["Status"] = pd.Series(assignments["Status"].to_numpy(), index=ids).reindex(frame.index).fillna("Durch Textfilter ausgeschlossen")
            frame["Dominanz"] = pd.Series(dominance, index=ids.loc[mask]).reindex(frame.index)
            frame["Mehrere Höchstwerte"] = pd.Series(ties, index=ids.loc[mask], dtype="boolean").reindex(frame.index).fillna(False)
            counts = frame["Topic"].value_counts().reindex(range(1, k + 1), fill_value=0)
            assigned = int(frame["Topic"].notna().sum())
            if int(counts.sum()) != assigned or assignments.loc[~mask, "Topic"].notna().any():
                raise RuntimeError("Die Häufigkeiten oder die Behandlung leerer Vektoren stimmen nicht.")
            common &= frame["Topic"].notna()
            for threshold in DIAG_THRESHOLDS:
                frame[f"Dominanz >= {threshold:.1f}"] = frame["Topic"].notna() & frame["Dominanz"].ge(threshold)
            meta = {"Variante": name, "Verfahren": "NMF", "Anzahl Topics": k, "min_df": vectorizer.min_df}
            model_rows.append({
                **meta, "Meldungen im Arbeitsdatensatz": len(originals),
                "Nach Text Preprocessing": len(docs), "Durch Textfilter ausgeschlossen": len(originals) - len(docs),
                "Für das Modelltraining": len(fit_docs), "Merkmale": len(features),
                "Zugeordnet": assigned, "Nicht zugeordnet": len(docs) - assigned,
                "Ohne Merkmale": int(frame["Status"].eq("Keine Merkmale im Vokabular").sum()),
                "Ohne positive Koeffizienten": int(frame["Status"].eq("Keine positiven Topic-Gewichte").sum()),
                "c_v Mittelwert": float(per_topic.mean()), "c_v Minimum": float(per_topic.min()),
                "Topics ohne Zuordnung": int(counts.eq(0).sum()), "Neu trainiert": not reused,
                "Iterationen": int(model.n_iter_), "Warnungen neuer Lauf": len(recorded),
                "Hinweise": " | ".join(str(w.message) for w in recorded),
            })
            for topic, terms in enumerate(topics, start=1):
                group = frame["Topic"].eq(topic).fillna(False)
                row = {**meta, "Topic": topic, "Top-Begriffe": ", ".join(terms),
                       "c_v": float(per_topic[topic - 1]), "Anzahl Meldungen": int(counts.loc[topic]),
                       "Anteil Prozent": 100 * int(counts.loc[topic]) / len(docs),
                       "Bezugsmenge Meldungen": len(docs)}
                for threshold in DIAG_THRESHOLDS:
                    kept = int((group & frame[f"Dominanz >= {threshold:.1f}"]).sum())
                    row[f"Anzahl Dominanz >= {threshold:.1f}"] = kept
                topic_rows.append(row)
            comparison_models[(name, k)] = {
                "model": model, "weights": coefficients, "assignments": assignments,
                "diagnostics": frame, "topics": topics,
            }
            print(f"  c_v: {per_topic.mean():.4f}, zugeordnet: {assigned}", flush=True)

    if not common.any():
        raise ValueError("Es verbleiben keine gemeinsam zugeordneten Meldungen für den Kategorienvergleich.")
    category_rows, dominance_rows, cross_rows, all_rows, sample_rows = [], [], [], [], []
    for (name, k), item in comparison_models.items():
        frame = item["diagnostics"]
        meta = {"Variante": name, "Verfahren": "NMF", "Anzahl Topics": k,
                "min_df": results[name]["tfidf_vectorizer"].min_df}
        assigned = frame["Topic"].notna()
        after = int(frame["Nach Text Preprocessing"].sum())
        for basis, chosen in (("Jeweils zugeordnet", assigned), ("Gemeinsam zugeordnet", common)):
            for column in DIAG_CATEGORIES:
                scores, table = category_statistics(frame, chosen, column)
                category_rows.append({**meta, "Datenbasis": basis, "Kategorieebene": column, **scores})
                for topic, values in table.iterrows():
                    cross_rows.extend({**meta, "Datenbasis": basis, "Kategorieebene": column,
                                       "Topic": int(topic), "Kategorie": category, "Anzahl": int(count)}
                                      for category, count in values.items())
        for threshold in DIAG_THRESHOLDS:
            high = frame[f"Dominanz >= {threshold:.1f}"]
            for label, chosen in (("Mindestens Schwelle", high), ("Unter Schwelle", assigned & ~high)):
                for column in DIAG_CATEGORIES:
                    scores, _ = category_statistics(frame, chosen, column)
                    dominance_rows.append({**meta, "Kategorieebene": column, "Schwelle": threshold,
                        "Teilmenge": label, **scores, "Nach Text Preprocessing": after,
                        "Anteil nach Preprocessing in Prozent": 100 * int(chosen.sum()) / after})
        exported = frame.copy()
        exported["Gemeinsam zugeordnet"] = common
        for column, value in reversed(list(meta.items())):
            exported.insert(0, column, value)
        all_rows.append(exported.reset_index())
        review = exported.loc[sample["ID"]].copy().reset_index()
        review.insert(0, "Prüffall", sample["Prüffall"].to_numpy())
        term_map = {i + 1: ", ".join(terms) for i, terms in enumerate(item["topics"])}
        review["Top-Begriffe"] = review["Topic"].map(term_map).fillna("")
        sample_rows.append(review)

    summary = pd.DataFrame(model_rows)
    categories = pd.DataFrame(category_rows)
    for level, suffix in (("category_parent", "Haupt"), ("category_child", "Unter")):
        selected = categories.loc[categories["Datenbasis"].eq("Gemeinsam zugeordnet") & categories["Kategorieebene"].eq(level)]
        fields = ["Bewertete Meldungen", "NMI", "AMI", "ARI", "Reinheit"]
        summary = summary.merge(selected[["Variante", "Anzahl Topics", *fields]].rename(
            columns={field: f"{field} {suffix}" for field in fields}),
            on=["Variante", "Anzahl Topics"], how="left", validate="one_to_one")
    tables = {
        "vergleich_nmf_topic_anzahl": summary,
        "topics_nmf_topic_anzahl": pd.DataFrame(topic_rows),
        "kategorien_nmf_topic_anzahl": categories,
        "kreuztabellen_nmf_topic_anzahl": pd.DataFrame(cross_rows),
        "dominanz_nmf_topic_anzahl": pd.DataFrame(dominance_rows),
        "zuordnungen_nmf_topic_anzahl": pd.concat(all_rows, ignore_index=True),
        "stichprobe_nmf_topic_anzahl": pd.concat(sample_rows, ignore_index=True),
    }
    if len(tables["stichprobe_nmf_topic_anzahl"]) != 100 * total:
        raise RuntimeError("Nicht alle festgelegten Fälle sind im Vergleich enthalten.")
    if before != diagnostic_fingerprint(protected):
        raise RuntimeError("Die Eingaben oder vorhandenen Modelle wurden durch den Vergleich verändert.")

    out = Path(output_dir) / "optimierung" / "nmf_topic_anzahl"
    out.mkdir(parents=True, exist_ok=True)
    paths = []
    for name, table in tables.items():
        path = out / f"{name}.csv"
        table.to_csv(path, index=False, sep=";", encoding="utf-8-sig", lineterminator="\n")
        paths.append(path)
    settings = {
        "zweck": "Modellgröße zweier fester Textvarianten vergleichen, keine Änderung der Hauptanalyse",
        "varianten": list(NMF_VARIANTS_TO_COMPARE), "topic_anzahlen": list(NMF_TOPIC_COUNTS),
        "c_v": {"topn": top_n, "window_size": window_size, "processes": 1,
                "texte": len(reference_texts), "basis": "Gleiche normalisierte Texte beider Varianten, ohne Vokabularfilter"},
        "texte_sha256": hashlib.sha256(reference.to_csv(index=False, lineterminator="\n").encode("utf-8")).hexdigest(),
        "gemeinsame_ids_11i": anchor_count, "gemeinsame_ids_vergleich": int(common.sum()),
        "gemeinsame_basis": "Schnittmenge aus 11i, zusätzlich auf Zuordnung in allen zehn Kandidaten geprüft",
        "schwellen": list(DIAG_THRESHOLDS), "vergleichsoperator": ">=", "nur_diagnose": True,
        "wiederverwendet": "Die beiden Modelle mit zwölf Topics aus 11h, keine erneuten Trainingswarnungen erfasst",
        "parameter": {name: results[name]["models"]["NMF"].get_params() for name in NMF_VARIANTS_TO_COMPARE},
        "eingaben_fingerabdruck": before, "einstellungen_11h": previous_settings,
        "versionen": {"numpy": np.__version__, "pandas": pd.__version__, "sklearn": sklearn.__version__, "gensim": gensim.__version__},
        "sha256_ausgaben": {path.name: hashlib.sha256(path.read_bytes()).hexdigest() for path in paths},
    }
    path = out / "einstellungen_nmf_topic_anzahl.json"
    path.write_text(json.dumps(settings, ensure_ascii=False, indent=2, default=str) + "\n", encoding="utf-8")
    paths.append(path)
    archive = out / "vergleich_nmf_topic_anzahl.zip"
    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as bundle:
        for path in paths:
            bundle.write(path, arcname=path.name)
    display(summary[["Variante", "Anzahl Topics", "c_v Mittelwert", "AMI Haupt", "AMI Unter", "ARI Haupt", "ARI Unter", "Zugeordnet"]].round(4))
    print("Verglichene NMF-Modelle:", len(summary))
    print("Neu trainiert:", int(summary["Neu trainiert"].sum()))
    print("Gemeinsame Meldungen für den Kategorienvergleich:", int(common.sum()))
    print("Technische Kontrolle des NMF-Vergleichs: bestanden")
    print("Die Hauptanalyse und ihre bisherigen Zuordnungen sind unverändert.")
    print("Gespeichert:", archive.resolve())
    return {"models": comparison_models, "tables": tables, "settings": settings, "zip_path": archive}


_required_nmf_counts = (
    "optimization_results", "optimization_settings", "category_diagnostics", "df_work",
    "shared_review_texts", "diagnostic_fingerprint", "diagnostic_ids", "diagnostic_frames",
    "category_statistics", "topic_dominance", "assign_topics", "normalize_nmf_for_assignment",
    "DIAG_CATEGORIES", "DIAG_THRESHOLDS", "OUTPUT_DIR", "ID_COLUMN", "TEXT_COLUMN",
)
_missing_nmf_counts = [name for name in _required_nmf_counts if name not in globals()]
if _missing_nmf_counts:
    raise RuntimeError("Zuerst bis einschließlich 11i ausführen. Es fehlen: " + ", ".join(_missing_nmf_counts))
nmf_size_comparison = compare_normalized_nmf_counts(
    optimization_results, category_diagnostics, df_work, shared_review_texts,
    optimization_settings, OUTPUT_DIR, ID_COLUMN, TEXT_COLUMN,
)


In [ ]:
# Schritt 11k Count und TF-IDF mit mehreren Initialisierungen vergleichen
# Der Kandidat wird separat geprüft, die Hauptanalyse bleibt unverändert.

from copy import deepcopy
from itertools import combinations
from pathlib import Path
import hashlib
import json
import warnings
import zipfile

import numpy as np
import pandas as pd
import sklearn
import gensim
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
from sklearn.base import clone
from sklearn.metrics import adjusted_rand_score

NMF_CHECK_VARIANT = "Kombination"
NMF_CHECK_TOPICS = 15
NMF_CHECK_SEEDS = (42, 7, 21, 84, 123)


def compare_nmf_vectorizers_and_seeds(results, size_comparison, output_dir, id_column):
    """Count und TF-IDF bei identischen Texten, Topic-Anzahl und Seeds prüfen."""
    key = (NMF_CHECK_VARIANT, NMF_CHECK_TOPICS)
    if NMF_CHECK_VARIANT not in results or key not in size_comparison["models"]:
        raise ValueError("Der Kandidat Kombination mit 15 Topics aus Schritt 11j fehlt.")
    if len(NMF_CHECK_SEEDS) != 5 or len(set(NMF_CHECK_SEEDS)) != 5 or 42 not in NMF_CHECK_SEEDS:
        raise ValueError("Erwartet werden fünf unterschiedliche Seeds einschließlich 42.")
    if any(isinstance(seed, bool) or not isinstance(seed, int) or seed < 0 for seed in NMF_CHECK_SEEDS):
        raise ValueError("Seeds müssen nicht negative ganze Zahlen sein.")

    data = results[NMF_CHECK_VARIANT]
    candidate = size_comparison["models"][key]
    protected = (results, size_comparison)
    before = diagnostic_fingerprint(protected)
    docs, fit_docs = data["documents"], data["fit_documents"]
    mask = np.asarray(data["mask"])
    ids = diagnostic_ids(docs[id_column])
    if mask.dtype != bool or mask.shape != (len(docs),) or not docs.index[mask].equals(fit_docs.index):
        raise ValueError("Meldungen und Trainingsauswahl passen nicht zusammen.")
    if not docs.loc[mask, [id_column, "text_clean"]].equals(fit_docs[[id_column, "text_clean"]]):
        raise ValueError("Die Texte der Trainingsauswahl wurden verändert.")
    vectorizers = (data["count_vectorizer"], data["tfidf_vectorizer"])
    features = vectorizers[0].get_feature_names_out()
    if not np.array_equal(features, vectorizers[1].get_feature_names_out()):
        raise ValueError("Count und TF-IDF benötigen dasselbe Vokabular.")
    if any(v.min_df != 2 or v.ngram_range != (1, 1) for v in vectorizers):
        raise ValueError("Der Vergleich benötigt die Kombination aus 11h mit min_df=2 und Unigrammen.")
    text_frame = docs[[id_column, "text_clean"]].copy()
    text_frame[id_column] = ids
    text_hash = hashlib.sha256(text_frame.to_csv(index=False, lineterminator="\n").encode("utf-8")).hexdigest()
    if text_hash != size_comparison["settings"]["texte_sha256"]:
        raise ValueError("Die Textgrundlage stimmt nicht mit Schritt 11j überein.")
    template = candidate["model"]
    if template.n_components != NMF_CHECK_TOPICS or template.random_state != 42:
        raise ValueError("Das vorhandene Modell entspricht nicht dem ausgewählten Kandidaten.")
    if template.init != "nndsvda":
        raise ValueError("Für diesen Vergleich bleibt die Initialisierung nndsvda erhalten.")
    matrices = {"Count": data["count_fit"], "TF-IDF": data["tfidf_fit"]}
    for name, matrix in matrices.items():
        if matrix.shape != (len(fit_docs), len(features)) or not np.isfinite(matrix.data).all() or (matrix.data < 0).any():
            raise ValueError(f"{name}: Merkmalsmatrix und Trainingsmeldungen passen nicht zusammen.")

    tokens = docs["text_clean"].str.split().tolist()
    dictionary = Dictionary(tokens)
    top_n, window_size = 10, 110
    runs, topic_rows = {}, []
    coherence_cache = {}
    jobs = [(name, seed) for name in ("TF-IDF", "Count") for seed in NMF_CHECK_SEEDS]
    for position, (name, seed) in enumerate(jobs, start=1):
        reused = name == "TF-IDF" and seed == 42
        print(f"[{position}/{len(jobs)}] NMF | {name} | Seed {seed} | "
              + ("Modell aus 11j" if reused else "neues Training"), flush=True)
        recorded = []
        if reused:
            model = deepcopy(template)
            coefficients = np.asarray(candidate["weights"], dtype=float).copy()
        else:
            model = clone(template).set_params(random_state=seed)
            with warnings.catch_warnings(record=True) as recorded:
                warnings.simplefilter("always")
                raw = model.fit_transform(matrices[name].copy())
            coefficients, _ = normalize_nmf_for_assignment(raw, model.components_)
        assignments = assign_topics(coefficients, mask, docs.index)
        if reused and not assignments.equals(candidate["assignments"]):
            raise RuntimeError("Die übernommenen Zuordnungen stimmen nicht mit 11j überein.")
        h = np.asarray(model.components_)
        if h.shape != (NMF_CHECK_TOPICS, len(features)) or ((h > 0).sum(axis=1) < top_n).any():
            raise RuntimeError("Mindestens ein Topic enthält zu wenige positive Begriffe für c_v.")
        topics = [features[index].tolist() for index in np.argsort(-h, axis=1, kind="stable")[:, :top_n]]
        cache_key = tuple(tuple(topic) for topic in topics)
        if cache_key not in coherence_cache:
            coherence_cache[cache_key] = np.asarray(CoherenceModel(
                topics=topics, texts=tokens, dictionary=dictionary, coherence="c_v",
                topn=top_n, window_size=window_size, processes=1,
            ).get_coherence_per_topic(), dtype=float)
        coherence = coherence_cache[cache_key]
        if not np.isfinite(coherence).all():
            raise RuntimeError("Die Kohärenzwerte sind nicht vollständig berechenbar.")
        if reused:
            prior = size_comparison["tables"]["vergleich_nmf_topic_anzahl"]
            prior = prior.loc[prior["Variante"].eq(NMF_CHECK_VARIANT) & prior["Anzahl Topics"].eq(NMF_CHECK_TOPICS)]
            if len(prior) != 1 or not np.isclose(coherence.mean(), prior.iloc[0]["c_v Mittelwert"], rtol=0, atol=1e-10):
                raise RuntimeError("Der wiederverwendete Kohärenzwert stimmt nicht mit Schritt 11j überein.")

        dominance, ties = topic_dominance(coefficients)
        frame = docs[["category_parent", "category_child"]].copy()
        for col in frame.columns:
            frame[col] = frame[col].astype("string").str.strip().replace("", pd.NA)
        frame["Topic"], frame["Status"] = assignments["Topic"], assignments["Status"]
        frame["Dominanz"] = pd.Series(dominance, index=fit_docs.index).reindex(docs.index)
        counts = assignments["Topic"].value_counts().reindex(range(1, NMF_CHECK_TOPICS + 1), fill_value=0)
        messages = [str(w.message) for w in recorded]
        for message in messages:
            print("  Warnung:", message)
        runs[(name, seed)] = {
            "model": model, "weights": coefficients, "assignments": assignments, "frame": frame,
            "coherence": coherence, "reused": reused, "warnings": messages,
        }
        for topic, terms in enumerate(topics, start=1):
            topic_rows.append({"Vektorisierung": name, "Seed": seed, "Anzahl Topics": NMF_CHECK_TOPICS,
                "Topic": topic, "Top-Begriffe": ", ".join(terms), "c_v": coherence[topic - 1],
                "Anzahl Meldungen": int(counts.loc[topic]), "Anteil Prozent": 100 * counts.loc[topic] / len(docs)})

    # Dieselben IDs wie in 11j, zusätzlich gemeinsame Zuordnung in allen neuen Läufen.
    previous = size_comparison["tables"]["zuordnungen_nmf_topic_anzahl"]
    common_ids = set(previous.loc[previous["Gemeinsam zugeordnet"], "ID"].astype("string"))
    common = pd.Series(ids.isin(common_ids).to_numpy(), index=docs.index)
    for run in runs.values():
        common &= run["assignments"]["Topic"].notna()
    if not common.any():
        raise ValueError("Es gibt keine gemeinsame Menge für den Kategorienvergleich.")

    rows, exported = [], []
    for (name, seed), run in runs.items():
        frame, model = run["frame"], run["model"]
        row = {"Vektorisierung": name, "Seed": seed, "Anzahl Topics": NMF_CHECK_TOPICS,
            "min_df": 2, "Nach Text Preprocessing": len(docs), "Für das Modelltraining": len(fit_docs),
            "Merkmale": len(features), "Zugeordnet": int(frame["Topic"].notna().sum()),
            "Nicht zugeordnet": int(frame["Topic"].isna().sum()),
            "c_v Mittelwert": float(run["coherence"].mean()), "c_v Minimum": float(run["coherence"].min()),
            "Neu trainiert": not run["reused"], "Iterationen": int(model.n_iter_),
            "Warnungen neuer Lauf": len(run["warnings"]), "Hinweise": " | ".join(run["warnings"])}
        for column, suffix in (("category_parent", "Haupt"), ("category_child", "Unter")):
            scores, _ = category_statistics(frame, common, column)
            for field in ("Bewertete Meldungen", "NMI", "AMI", "ARI", "Reinheit"):
                row[f"{field} {suffix}"] = scores[field]
        rows.append(row)
        part = frame[["Topic", "Status", "Dominanz"]].copy()
        part.insert(0, "ID", ids.to_numpy())
        part.insert(0, "Seed", seed)
        part.insert(0, "Vektorisierung", name)
        part["Gemeinsam zugeordnet"] = common
        exported.append(part.reset_index(drop=True))
    pair_rows = []
    for name in ("TF-IDF", "Count"):
        for first, second in combinations(NMF_CHECK_SEEDS, 2):
            left, right = runs[(name, first)]["assignments"], runs[(name, second)]["assignments"]
            chosen = left["Topic"].notna() & right["Topic"].notna()
            pair_rows.append({"Vektorisierung": name, "Seed 1": first, "Seed 2": second,
                "Verglichene Meldungen": int(chosen.sum()),
                "ARI zwischen Initialisierungen": adjusted_rand_score(left.loc[chosen, "Topic"].astype(int), right.loc[chosen, "Topic"].astype(int)),
                "Unterschiedlicher Zuordnungsstatus": int(left["Topic"].notna().ne(right["Topic"].notna()).sum())})
    all_runs = pd.DataFrame(rows)
    group_rows = []
    for name, part in all_runs.groupby("Vektorisierung", sort=False):
        summary = {"Vektorisierung": name, "Läufe": len(part), "Anzahl Topics": NMF_CHECK_TOPICS}
        for metric in ("c_v Mittelwert", "AMI Haupt", "AMI Unter", "ARI Haupt", "ARI Unter"):
            summary[f"{metric} Mittel"] = float(part[metric].mean())
            summary[f"{metric} Standardabweichung"] = float(part[metric].std(ddof=1))
        group_rows.append(summary)
    tables = {"vergleich_count_tfidf_lauefe": all_runs,
        "vergleich_count_tfidf_zusammenfassung": pd.DataFrame(group_rows),
        "stabilitaet_nmf_paare": pd.DataFrame(pair_rows),
        "topics_count_tfidf": pd.DataFrame(topic_rows),
        "zuordnungen_count_tfidf": pd.concat(exported, ignore_index=True)}
    if diagnostic_fingerprint(protected) != before:
        raise RuntimeError("Vorhandene Daten oder Modelle wurden durch den Vergleich verändert.")

    folder = Path(output_dir) / "optimierung" / "vektorisierung_stabilitaet"
    folder.mkdir(parents=True, exist_ok=True)
    paths = []
    for name, table in tables.items():
        path = folder / f"{name}.csv"
        table.to_csv(path, sep=";", index=False, encoding="utf-8-sig", lineterminator="\n")
        paths.append(path)
    settings = {"zweck": "Count versus TF-IDF und Stabilität, keine Änderung der Hauptanalyse",
        "variante": NMF_CHECK_VARIANT, "anzahl_topics": NMF_CHECK_TOPICS, "seeds": list(NMF_CHECK_SEEDS),
        "texte_sha256": text_hash, "gemeinsame_meldungen": int(common.sum()),
        "parameter_nmf": template.get_params(), "parameter_count": vectorizers[0].get_params(),
        "parameter_tfidf": vectorizers[1].get_params(),
        "c_v": {"topn": top_n, "window_size": window_size, "processes": 1, "texte": len(tokens)},
        "skalierung": "Count unverändert, TF-IDF wie in 11h, NMF-Koeffizienten jeweils auf L2-normierte Topic-Vektoren abgestimmt.",
        "stabilitaet": "Paarweiser ARI zwischen Seeds innerhalb derselben Vektorisierung, keine inhaltliche Trefferquote.",
        "wiederverwendet": "TF-IDF mit Seed 42 aus 11j. Keine neuen Trainingswarnungen für dieses Modell erfasst.",
        "versionen": {"numpy": np.__version__, "pandas": pd.__version__, "sklearn": sklearn.__version__, "gensim": gensim.__version__},
        "eingaben_fingerabdruck": before,
        "sha256_ausgaben": {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in paths}}
    settings_path = folder / "einstellungen_vektorisierung_stabilitaet.json"
    settings_path.write_text(json.dumps(settings, ensure_ascii=False, indent=2, default=str) + "\n", encoding="utf-8")
    paths.append(settings_path)
    archive = folder / "vergleich_vektorisierung_stabilitaet.zip"
    with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
        for path in paths:
            bundle.write(path, arcname=path.name)
    display(all_runs[["Vektorisierung", "Seed", "c_v Mittelwert", "AMI Haupt", "AMI Unter", "Iterationen", "Warnungen neuer Lauf"]].round(4))
    print("Verglichene NMF-Läufe:", len(all_runs))
    print("Neu trainiert:", int(all_runs["Neu trainiert"].sum()))
    print("Gemeinsame Meldungen für den Kategorienvergleich:", int(common.sum()))
    print("Technische Kontrolle von Vektorisierung und Stabilität: bestanden")
    print("Die Hauptanalyse und ihre bisherigen Zuordnungen sind unverändert.")
    print("Gespeichert:", archive.resolve())
    return {"models": runs, "tables": tables, "settings": settings, "zip_path": archive}


_required_nmf_check = ("optimization_results", "nmf_size_comparison", "diagnostic_fingerprint", "diagnostic_ids",
    "category_statistics", "topic_dominance", "assign_topics", "normalize_nmf_for_assignment", "OUTPUT_DIR", "ID_COLUMN")
_missing_nmf_check = [name for name in _required_nmf_check if name not in globals()]
if _missing_nmf_check:
    raise RuntimeError("Zuerst bis einschließlich Schritt 11j ausführen. Es fehlen: " + ", ".join(_missing_nmf_check))
nmf_vectorizer_check = compare_nmf_vectorizers_and_seeds(
    optimization_results, nmf_size_comparison, OUTPUT_DIR, ID_COLUMN,
)


In [ ]:
# Schritt 11l Trainingsumfang und Initialisierung bei LDA vergleichen
# Die vorhandenen Textvarianten, Modelle und Ergebnisübersichten bleiben unverändert.

from copy import deepcopy
from itertools import combinations
from pathlib import Path
import hashlib
import json
import warnings
import zipfile

import numpy as np
import pandas as pd
import sklearn
import gensim
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel
from sklearn.base import clone
from sklearn.metrics import adjusted_rand_score

LDA_CHECK_VARIANTS = ("Referenz", "Kombination")
LDA_CHECK_ITERATIONS = (20, 100)
LDA_CHECK_SEEDS = (42, 7, 21, 84, 123)


def compare_lda_training_and_seeds(results, diagnostics, previous_settings, output_dir, id_column):
    """Zwei Trainingslimits bei gleicher Topic-Anzahl und fünf festgelegten Seeds vergleichen."""
    if LDA_CHECK_VARIANTS != ("Referenz", "Kombination") or LDA_CHECK_ITERATIONS != (20, 100):
        raise ValueError("Dieser Vergleich ist für Referenz und Kombination mit 20 und 100 Iterationen vorgesehen.")
    if (len(LDA_CHECK_SEEDS) != 5 or len(set(LDA_CHECK_SEEDS)) != 5 or 42 not in LDA_CHECK_SEEDS
            or any(isinstance(s, bool) or not isinstance(s, int) or s < 0 for s in LDA_CHECK_SEEDS)):
        raise ValueError("Erwartet werden fünf unterschiedliche, nicht negative ganzzahlige Seeds einschließlich 42.")
    if any(name not in results for name in LDA_CHECK_VARIANTS):
        raise ValueError("Die benötigten Varianten aus Schritt 11h fehlen.")
    if diagnostic_fingerprint(diagnostics["einstellungen"]["einstellungen_11h"]) != diagnostic_fingerprint(previous_settings):
        raise ValueError("Schritt 11h und 11i beziehen sich auf unterschiedliche Einstellungen.")
    protected = (results, diagnostics, previous_settings)
    before = diagnostic_fingerprint(protected)

    # Die Kategorien werden weiterhin auf denselben IDs wie in Schritt 11i verglichen.
    prior = diagnostics["dominanz_meldungen"]
    common_ids = set(prior.loc[prior["Gemeinsam zugeordnet"], "ID"].astype("string"))
    if not common_ids or len(common_ids) != diagnostics["einstellungen"]["gemeinsam_zugeordnet"]:
        raise ValueError("Die gemeinsame Menge aus Schritt 11i ist nicht eindeutig nachvollziehbar.")

    runs, rows, topic_rows, exported, variant_settings = {}, [], [], [], {}
    jobs = [(name, limit, seed) for name in LDA_CHECK_VARIANTS
            for limit in LDA_CHECK_ITERATIONS for seed in LDA_CHECK_SEEDS]
    for variant in LDA_CHECK_VARIANTS:
        data = results[variant]
        docs, fit_docs = data["documents"], data["fit_documents"]
        ids = diagnostic_ids(docs[id_column])
        mask = np.asarray(data["mask"])
        if (not docs.index.is_unique or mask.dtype != bool or mask.shape != (len(docs),)
                or not docs.index[mask].equals(fit_docs.index)
                or not docs.loc[mask, [id_column, "text_clean"]].equals(fit_docs[[id_column, "text_clean"]])):
            raise ValueError(f"{variant}: Meldungen und Trainingsauswahl passen nicht zusammen.")
        common = pd.Series(ids.isin(common_ids).to_numpy(), index=docs.index)
        if int(common.sum()) != len(common_ids) or not np.all(mask[common.to_numpy()]):
            raise ValueError(f"{variant}: Die gemeinsame Menge enthält Meldungen ohne Merkmale.")
        vectorizer, template = data["count_vectorizer"], data["models"]["LDA"]
        features = vectorizer.get_feature_names_out()
        matrix = data["count_fit"]
        expected_min_df = 5 if variant == "Referenz" else 2
        if vectorizer.min_df != expected_min_df or vectorizer.ngram_range != (1, 1):
            raise ValueError(f"{variant}: Das Vokabular entspricht nicht Schritt 11h.")
        if (template.n_components != 10 or template.max_iter != 20 or template.random_state != 42
                or template.learning_method != "batch" or template.evaluate_every > 0
                or template.doc_topic_prior != 0.1 or template.topic_word_prior != 0.1):
            raise ValueError(f"{variant}: Erwartet wird LDA/10 aus 11h mit 20 Iterationen und Priors 0.1/0.1.")
        if (matrix.shape != (len(fit_docs), len(features)) or not np.isfinite(matrix.data).all()
                or (matrix.data < 0).any() or (matrix.getnnz(axis=1) == 0).any()):
            raise ValueError(f"{variant}: Die Count-Matrix ist ungültig.")
        tokens = docs["text_clean"].str.split().tolist()
        if any(not words for words in tokens):
            raise ValueError(f"{variant}: Die Referenztexte für c_v enthalten einen leeren Text.")
        dictionary, coherence_cache = Dictionary(tokens), {}
        text_frame = docs[[id_column, "text_clean"]].copy()
        text_frame[id_column] = ids
        variant_settings[variant] = {
            "parameter_lda": template.get_params(), "parameter_count": vectorizer.get_params(),
            "texte_sha256": hashlib.sha256(text_frame.to_csv(index=False, lineterminator="\n").encode("utf-8")).hexdigest(),
            "nach_preprocessing": len(docs), "training": len(fit_docs), "merkmale": len(features),
        }

        for limit in LDA_CHECK_ITERATIONS:
            for seed in LDA_CHECK_SEEDS:
                reused = limit == 20 and seed == 42
                print(f"[{len(rows) + 1}/{len(jobs)}] LDA | {variant} | {limit} Iterationen | Seed {seed} | "
                      + ("Modell aus 11h" if reused else "neues Training"), flush=True)
                recorded = []
                if reused:
                    model = deepcopy(template)
                    coefficients = np.asarray(data["weights"]["LDA"], dtype=float).copy()
                else:
                    model = clone(template).set_params(max_iter=limit, random_state=seed)
                    with warnings.catch_warnings(record=True) as recorded:
                        warnings.simplefilter("always")
                        coefficients = model.fit_transform(matrix.copy())
                if (coefficients.shape != (len(fit_docs), 10) or not np.isfinite(coefficients).all()
                        or (coefficients < 0).any() or not np.allclose(coefficients.sum(axis=1), 1.0)):
                    raise RuntimeError("Die LDA-Themenanteile sind ungültig.")
                assignments = assign_topics(coefficients, mask, docs.index)
                if reused and not assignments.equals(data["assignments"]["LDA"]):
                    raise RuntimeError("Die übernommenen LDA-Zuordnungen stimmen nicht mit Schritt 11h überein.")
                if assignments.loc[common, "Topic"].isna().any():
                    raise RuntimeError("Ein Modell kann nicht auf der vollständigen gemeinsamen Menge bewertet werden.")
                components = np.asarray(model.components_)
                if components.shape != (10, len(features)) or not np.isfinite(components).all() or (components <= 0).any():
                    raise RuntimeError("Die LDA-Komponenten passen nicht zum Vokabular.")
                topics = [features[index].tolist() for index in np.argsort(-components, axis=1, kind="stable")[:, :10]]
                key = tuple(tuple(words) for words in topics)
                if key not in coherence_cache:
                    coherence_cache[key] = np.asarray(CoherenceModel(
                        topics=topics, texts=tokens, dictionary=dictionary, coherence="c_v",
                        topn=10, window_size=110, processes=1,
                    ).get_coherence_per_topic(), dtype=float)
                coherence = coherence_cache[key]
                # Perplexität nur auf derselben Textvariante und Count-Matrix vergleichen.
                # Sie bewertet die Anpassung an Trainingsdaten, nicht die richtige Interpretation.
                perplexity = float(model.perplexity(matrix.copy()))
                if not np.isfinite(coherence).all() or not np.isfinite(perplexity) or perplexity <= 0:
                    raise RuntimeError("Kohärenz oder Trainingsperplexität sind nicht berechenbar.")
                dominance, _ = topic_dominance(coefficients)
                frame = docs[["category_parent", "category_child"]].copy()
                for col in frame.columns:
                    frame[col] = frame[col].astype("string").str.strip().replace("", pd.NA)
                frame["Topic"], frame["Status"] = assignments["Topic"], assignments["Status"]
                frame["Dominanz"] = pd.Series(dominance, index=fit_docs.index).reindex(docs.index)
                counts = assignments["Topic"].value_counts().reindex(range(1, 11), fill_value=0)
                messages = [str(item.message) for item in recorded]
                for message in messages:
                    print("  Warnung:", message)
                row = {
                    "Variante": variant, "max_iter": limit, "Seed": seed, "Anzahl Topics": 10,
                    "min_df": vectorizer.min_df, "Nach Text Preprocessing": len(docs),
                    "Für das Modelltraining": len(fit_docs), "Merkmale": len(features),
                    "Zugeordnet": int(assignments["Topic"].notna().sum()),
                    "Nicht zugeordnet": int(assignments["Topic"].isna().sum()),
                    "c_v Mittelwert": float(coherence.mean()), "c_v Minimum": float(coherence.min()),
                    "Perplexität Training": perplexity, "Iterationen": int(model.n_iter_),
                    "Neu trainiert": not reused, "Warnungen neuer Lauf": len(recorded), "Hinweise": " | ".join(messages),
                }
                for column, suffix in (("category_parent", "Haupt"), ("category_child", "Unter")):
                    scores, _ = category_statistics(frame, common, column)
                    for field in ("Bewertete Meldungen", "NMI", "AMI", "ARI", "Reinheit"):
                        row[f"{field} {suffix}"] = scores[field]
                rows.append(row)
                for topic, words in enumerate(topics, start=1):
                    topic_rows.append({"Variante": variant, "max_iter": limit, "Seed": seed,
                        "Topic": topic, "Top-Begriffe": ", ".join(words), "c_v": float(coherence[topic - 1]),
                        "Anzahl Meldungen": int(counts.loc[topic]), "Anteil Prozent": 100 * counts.loc[topic] / len(docs)})
                out = frame[["Topic", "Status", "Dominanz"]].copy()
                out.insert(0, "ID", ids.to_numpy())
                for name, value in (("Seed", seed), ("max_iter", limit), ("Variante", variant)):
                    out.insert(0, name, value)
                out["Gemeinsam zugeordnet"] = common
                exported.append(out.reset_index(drop=True))
                runs[(variant, limit, seed)] = {"model": model, "weights": coefficients,
                    "assignments": assignments, "frame": frame, "coherence": coherence, "perplexity": perplexity}

    all_runs = pd.DataFrame(rows)
    summaries, paired_limits, paired_seeds = [], [], []
    for (variant, limit), part in all_runs.groupby(["Variante", "max_iter"], sort=False):
        row = {"Variante": variant, "max_iter": limit, "Läufe": len(part), "Anzahl Topics": 10}
        for metric in ("c_v Mittelwert", "Perplexität Training", "AMI Haupt", "AMI Unter", "ARI Haupt", "ARI Unter"):
            row[f"{metric} Mittel"] = float(part[metric].mean())
            row[f"{metric} Standardabweichung"] = float(part[metric].std(ddof=1))
        summaries.append(row)
        for seed_a, seed_b in combinations(LDA_CHECK_SEEDS, 2):
            left, right = runs[(variant, limit, seed_a)]["assignments"], runs[(variant, limit, seed_b)]["assignments"]
            chosen = left["Topic"].notna() & right["Topic"].notna()
            paired_seeds.append({"Variante": variant, "max_iter": limit, "Seed 1": seed_a, "Seed 2": seed_b,
                "Verglichene Meldungen": int(chosen.sum()),
                "ARI zwischen Initialisierungen": adjusted_rand_score(left.loc[chosen, "Topic"].astype(int), right.loc[chosen, "Topic"].astype(int))})
    for variant in LDA_CHECK_VARIANTS:
        for seed in LDA_CHECK_SEEDS:
            first, second = [runs[(variant, limit, seed)] for limit in LDA_CHECK_ITERATIONS]
            left, right = first["assignments"], second["assignments"]
            chosen = left["Topic"].notna() & right["Topic"].notna()
            paired_limits.append({"Variante": variant, "Seed": seed, "Iteration A": 20, "Iteration B": 100,
                "Verglichene Meldungen": int(chosen.sum()),
                "ARI 20 gegen 100": adjusted_rand_score(left.loc[chosen, "Topic"].astype(int), right.loc[chosen, "Topic"].astype(int)),
                "Perplexität 20": first["perplexity"], "Perplexität 100": second["perplexity"],
                "Änderung Perplexität in Prozent": 100 * (second["perplexity"] / first["perplexity"] - 1)})
    tables = {"vergleich_lda_lauefe": all_runs, "vergleich_lda_zusammenfassung": pd.DataFrame(summaries),
        "lda_iterationen_paare": pd.DataFrame(paired_limits), "lda_stabilitaet_paare": pd.DataFrame(paired_seeds),
        "topics_lda_training": pd.DataFrame(topic_rows), "zuordnungen_lda_training": pd.concat(exported, ignore_index=True)}
    if len(all_runs) != 20 or int(all_runs["Neu trainiert"].sum()) != 18:
        raise RuntimeError("Der Vergleich enthält nicht die vorgesehenen 20 Modelle und 18 neuen Trainingsläufe.")
    if diagnostic_fingerprint(protected) != before:
        raise RuntimeError("Die vorhandenen Daten oder Modelle wurden durch den Vergleich verändert.")
    folder = Path(output_dir) / "optimierung" / "lda_training_stabilitaet"
    folder.mkdir(parents=True, exist_ok=True)
    paths = []
    for name, table in tables.items():
        path = folder / f"{name}.csv"
        table.to_csv(path, sep=";", index=False, encoding="utf-8-sig", lineterminator="\n")
        paths.append(path)
    settings = {
        "zweck": "LDA/10 bei 20 und 100 Iterationen und fünf Seeds, keine Änderung der Hauptanalyse",
        "varianten": variant_settings, "iterationslimits": list(LDA_CHECK_ITERATIONS), "seeds": list(LDA_CHECK_SEEDS),
        "gemeinsame_meldungen": len(common_ids),
        "gemeinsame_ids_sha256": hashlib.sha256("\n".join(sorted(common_ids)).encode("utf-8")).hexdigest(),
        "coherence": "c_v, topn=10, window_size=110, processes=1. Jede Textvariante hat ihre eigenen Referenztexte.",
        "vergleich": "Perplexität und c_v vor allem innerhalb derselben Textvariante vergleichen. Kategorienmaße auf gemeinsamen IDs.",
        "perplexitaet": "Nach dem Fit auf der jeweiligen Trainingsmatrix berechnet, keine Testgenauigkeit oder Konvergenzgarantie.",
        "trainingslimit": "evaluate_every bleibt unverändert <= 0. Keine zusätzliche Frühstoppregel, keine Konvergenzbehauptung.",
        "wiederverwendet": "Je Variante LDA/10, 20 Iterationen, Seed 42 aus 11h. Null neue Warnungen ist dafür kein erneuter Trainingstest.",
        "bewertung": "Keine neue qualitative Bewertung und keine neue Stichprobe.",
        "eingaben_fingerabdruck": before,
        "versionen": {"numpy": np.__version__, "pandas": pd.__version__, "sklearn": sklearn.__version__, "gensim": gensim.__version__},
        "sha256_ausgaben": {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in paths},
    }
    path = folder / "einstellungen_lda_training_stabilitaet.json"
    path.write_text(json.dumps(settings, ensure_ascii=False, indent=2, default=str) + "\n", encoding="utf-8")
    paths.append(path)
    archive = folder / "vergleich_lda_training_stabilitaet.zip"
    with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:
        for path in paths:
            bundle.write(path, arcname=path.name)
    print(pd.DataFrame(summaries)[["Variante", "max_iter", "Läufe", "c_v Mittelwert Mittel", "Perplexität Training Mittel", "AMI Haupt Mittel", "AMI Unter Mittel"]].round(4).to_string(index=False))
    print("Verglichene LDA-Modelle:", len(all_runs))
    print("Neu trainiert:", int(all_runs["Neu trainiert"].sum()))
    print("Gemeinsame Meldungen für den Kategorienvergleich:", len(common_ids))
    print("Technische Kontrolle des LDA-Vergleichs: bestanden")
    print("Die Hauptanalyse und ihre bisherigen Zuordnungen sind unverändert.")
    print("Gespeichert:", archive.resolve())
    return {"models": runs, "tables": tables, "settings": settings, "zip_path": archive}


_required_lda_check = ("optimization_results", "category_diagnostics", "optimization_settings", "diagnostic_fingerprint",
    "diagnostic_ids", "category_statistics", "topic_dominance", "assign_topics", "OUTPUT_DIR", "ID_COLUMN")
_missing_lda_check = [name for name in _required_lda_check if name not in globals()]
if _missing_lda_check:
    raise RuntimeError("Zuerst die Vorstufen bis 11i ausführen. Es fehlen: " + ", ".join(_missing_lda_check))
lda_training_check = compare_lda_training_and_seeds(
    optimization_results, category_diagnostics, optimization_settings, OUTPUT_DIR, ID_COLUMN,
)


In [ ]:
# Schritt 12 Wichtigste Begriffe je Topic anzeigen

def print_top_words(model, feature_names, n_top_words=12, title=""):
    print("\n" + title)
    print("=" * len(title))

    for topic_idx, topic in enumerate(model.components_):
        top_indices = topic.argsort()[::-1][:n_top_words]
        top_words = [feature_names[i] for i in top_indices]
        print(f"Topic {topic_idx + 1}: {', '.join(top_words)}")

print_top_words(
    lda_model,
    count_vectorizer.get_feature_names_out(),
    n_top_words=12,
    title="LDA Topics auf CountVectorizer"
)

print_top_words(
    nmf_model,
    tfidf_vectorizer.get_feature_names_out(),
    n_top_words=12,
    title="NMF Topics auf TfidfVectorizer"
)


In [ ]:
# Schritt 13 Technische Themenübersichten erstellen

def build_topic_summary(model, feature_names, assignments, source_df, n_top_words=12, n_examples=2):
    if not assignments.index.equals(source_df.index):
        raise ValueError("Meldungen und Topic-Zuordnungen haben nicht denselben Index.")
    n_docs = len(source_df)
    if n_docs == 0:
        raise ValueError("Die Bezugsmenge für die Prozentanteile ist leer.")

    rows = []

    for topic_idx in range(model.n_components):
        topic_number = topic_idx + 1

        # Top Begriffe
        topic_weights = model.components_[topic_idx]
        top_indices = topic_weights.argsort()[::-1][:n_top_words]
        top_words = ", ".join(feature_names[top_indices])

        # Dokumente mit dominantem Topic
        doc_indices = np.flatnonzero(
            assignments["Topic"].eq(topic_number).fillna(False).to_numpy(dtype=bool)
        )
        doc_count = len(doc_indices)
        doc_share = round((doc_count / n_docs) * 100, 1)

        # Häufigste bestehende Kategorie zur Plausibilisierung
        if doc_count > 0:
            category_counts = (
                source_df.iloc[doc_indices]["category_parent"]
                .value_counts()
                .head(3)
            )
            top_categories = " | ".join(
                [f"{cat}: {count}" for cat, count in category_counts.items()]
            )

            # Erste zugeordnete Meldungen auswählen und kürzen.
            examples = " || ".join(
                source_df.iloc[doc_indices][TEXT_COLUMN]
                .astype(str)
                .head(n_examples)
                .str.slice(0, 180)
                .tolist()
            )
        else:
            top_categories = ""
            examples = ""

        rows.append({
            "Topic": topic_number,
            "Anzahl Meldungen": doc_count,
            "Anteil Prozent": doc_share,
            "Bezugsmenge Meldungen": n_docs,
            "Top Begriffe": top_words,
            "Häufigste Kategorie": top_categories,
            "Beispielmeldungen": examples
        })

    summary = pd.DataFrame(rows)
    summary = summary.sort_values("Anzahl Meldungen", ascending=False).reset_index(drop=True)
    summary.insert(0, "Rang", np.arange(1, len(summary) + 1))

    if int(summary["Anzahl Meldungen"].sum()) != int(assignments["Topic"].notna().sum()):
        raise RuntimeError("Die Themenhäufigkeiten stimmen nicht mit den Zuordnungen überein.")
    return summary

lda_summary = build_topic_summary(
    lda_model,
    count_vectorizer.get_feature_names_out(),
    lda_assignments,
    df_model
)

nmf_summary = build_topic_summary(
    nmf_model,
    tfidf_vectorizer.get_feature_names_out(),
    nmf_assignments,
    df_model
)

print("LDA Themenübersicht:")
display(lda_summary)

print("NMF Themenübersicht:")
display(nmf_summary)

print("Die Anteile beziehen sich auf alle", N_ANALYSIS_DOCS, "Meldungen nach Text Preprocessing.")
print("Nicht zugeordnete Meldungen sind in der separaten Zuordnungsübersicht enthalten.")
print("Gerundete Prozentanteile können sich geringfügig von 100 Prozent unterscheiden.")


In [ ]:
# Schritt 14 Technische Themenübersichten als CSV speichern



# Die Spalte mit Rohtextbeispielen nicht in diesen CSV-Export übernehmen.
# Beispielmeldungen können Adressen oder personenbezogene Hinweise enthalten.
lda_export = lda_summary.drop(columns=["Beispielmeldungen"], errors="ignore")
nmf_export = nmf_summary.drop(columns=["Beispielmeldungen"], errors="ignore")

lda_path = OUTPUT_DIR / "themenuebersicht_lda.csv"
nmf_path = OUTPUT_DIR / "themenuebersicht_nmf.csv"

lda_export.to_csv(lda_path, index=False, encoding="utf-8-sig")
nmf_export.to_csv(nmf_path, index=False, encoding="utf-8-sig")

print("Gespeichert:")
print(lda_path.resolve())
print(nmf_path.resolve())

assignment_path = OUTPUT_DIR / "zuordnung_lda_nmf.csv"
assignment_summary.to_csv(assignment_path, index=False, sep=";", encoding="utf-8-sig")
print(assignment_path.resolve())


In [ ]:
# Schritt 15 Die ausgewählten NMF-Topics beschriften

import numpy as np
import pandas as pd

# Die Angaben beziehen sich auf die geprüfte Stichprobe des Modells mit zwölf Topics.
# Die Beispiele sind sinngemäß zusammengefasst, keine wörtlichen Zitate.
# Gemischte Topics behalten ihre Nummer und erhalten keine zu enge Themenbezeichnung.
topic_descriptions = {
    1: {
        "Thema": "Beleuchtung",
        "Einordnung": "Thematischer Schwerpunkt",
        "Kurzbeschreibung": "Ausfälle und Schäden an der öffentlichen Beleuchtung.",
        "Beispielhafte Hinweise": "Laterne ausgefallen, Straßenlampe defekt",
        "Erwartete Begriffe": ("beleuchtung", "kurve", "licht"),
    },
    2: {
        "Thema": "Müll und Abfall",
        "Einordnung": "Thematischer Schwerpunkt",
        "Kurzbeschreibung": "Abgelagerter Müll und Abfälle an überfüllten Behältern.",
        "Beispielhafte Hinweise": "Müll abgestellt, Mülltonnen überfüllt",
        "Erwartete Begriffe": ("müll", "illegaler", "mülleimer"),
    },
    3: {
        "Thema": "Abgestellte Fahrräder",
        "Einordnung": "Thematischer Schwerpunkt",
        "Kurzbeschreibung": "Abgestellte, beschädigte oder längere Zeit ungenutzte Fahrräder.",
        "Beispielhafte Hinweise": "Defekte Fahrräder abgestellt, Fahrradrahmen gebrochen",
        "Erwartete Begriffe": ("fahrrad", "herrenloses", "sattel"),
    },
    4: {
        "Thema": "Gemischte Meldungen (Topic 4)",
        "Einordnung": "Gemischtes Topic",
        "Kurzbeschreibung": "Verschiedene Anliegen, darunter Müll, Beleuchtung und Straßenschäden.",
        "Beispielhafte Hinweise": "Möbel an der Straße, Laternen ausgefallen, Frostschäden",
        "Erwartete Begriffe": ("straße", "entlang", "abgemeldeter"),
    },
    5: {
        "Thema": "Gehwege und Einschränkungen",
        "Einordnung": "Thematischer Schwerpunkt",
        "Kurzbeschreibung": "Eingeschränkte Nutzung von Gehwegen durch Bewuchs, Gegenstände oder Sperrungen.",
        "Beispielhafte Hinweise": "Gehweg zugewachsen, Durchgang verengt, Baustelle auf dem Gehweg",
        "Erwartete Begriffe": ("gehweg", "wächst", "hecke"),
    },
    6: {
        "Thema": "Beschilderung",
        "Einordnung": "Thematischer Schwerpunkt",
        "Kurzbeschreibung": "Beschädigte, unleserliche oder umgestürzte Schilder.",
        "Beispielhafte Hinweise": "Schild umgeknickt, Straßenschild unkenntlich",
        "Erwartete Begriffe": ("schild", "umgefahren", "ausgeblichen"),
    },
    7: {
        "Thema": "Gemischte Meldungen (Topic 7)",
        "Einordnung": "Gemischtes Topic",
        "Kurzbeschreibung": "Scherben und weitere Anliegen zu Wegen, Pollern und Abdeckungen.",
        "Beispielhafte Hinweise": "Glasscherben auf dem Radweg, Poller herausgerissen, Gullydeckel lose",
        "Erwartete Begriffe": ("glasscherben", "radweg", "fahrradweg"),
    },
    8: {
        "Thema": "Ampeln",
        "Einordnung": "Thematischer Schwerpunkt",
        "Kurzbeschreibung": "Störungen von Ampeln und ihrer Bedienelemente.",
        "Beispielhafte Hinweise": "Ampeltaste defekt, Grünsignal bleibt aus",
        "Erwartete Begriffe": ("ampel", "fußgänger", "grün"),
    },
    9: {
        "Thema": "Gemischte Meldungen (Topic 9)",
        "Einordnung": "Gemischtes Topic",
        "Kurzbeschreibung": "Anliegen zu Bäumen, Gegenständen und abgestellten Fahrrädern.",
        "Beispielhafte Hinweise": "Baum auf dem Weg, Warnbarke unter einem Baum, Fahrrad ohne Sattel",
        "Erwartete Begriffe": ("baum", "quer", "umgestürzt"),
    },
    10: {
        "Thema": "Abgestellte Fahrzeuge",
        "Einordnung": "Thematischer Schwerpunkt",
        "Kurzbeschreibung": "Abgestellte Roller und andere Fahrzeuge, teilweise ohne Kennzeichen.",
        "Beispielhafte Hinweise": "Roller längere Zeit abgestellt, Fahrzeug ohne Kennzeichen",
        "Erwartete Begriffe": ("roller", "versicherungsschild", "gueltiges"),
    },
    11: {
        "Thema": "Gemischte Meldungen (Topic 11)",
        "Einordnung": "Gemischtes Topic",
        "Kurzbeschreibung": "Gegenstände im Wasser, Verschmutzungen und weitere unterschiedliche Anliegen.",
        "Beispielhafte Hinweise": "Abfall im Wasser, rutschige Treppe, abgestelltes Fahrzeug",
        "Erwartete Begriffe": ("wasser", "verstopft", "läuft"),
    },
    12: {
        "Thema": "Gemischte Meldungen (Topic 12)",
        "Einordnung": "Gemischtes Topic",
        "Kurzbeschreibung": "Meldungen zu Spielplätzen, Abfallbehältern, Müll und Wegschäden.",
        "Beispielhafte Hinweise": "Spielzeug defekt, Müllsäcke abgestellt, Schlaglöcher am Spielplatz",
        "Erwartete Begriffe": ("spielplatz", "vermuellt", "mülleimer"),
    },
}


def build_decision_overview(model, vocabulary, assignments, documents, summary, descriptions):
    """Beschriftungen mit den unveränderten Häufigkeiten der Modellzuordnung verbinden."""
    required = {"Topic", "Anzahl Meldungen", "Anteil Prozent", "Bezugsmenge Meldungen"}
    if not required.issubset(summary.columns):
        raise ValueError("In der technischen Themenübersicht fehlen benötigte Spalten.")
    if len(documents) == 0 or not documents.index.is_unique:
        raise ValueError("Die Meldungen müssen einen nicht leeren, eindeutigen Index haben.")
    if not assignments.index.equals(documents.index):
        raise ValueError("Meldungen und Topic-Zuordnungen passen nicht zusammen.")
    if model.n_components != 12 or set(descriptions) != set(range(1, 13)):
        raise ValueError("Diese Beschriftungen gehören zum ausgewählten Modell mit zwölf Topics.")
    if summary["Topic"].duplicated().any() or set(summary["Topic"]) != set(descriptions):
        raise ValueError("Die technische Übersicht muss jedes Topic genau einmal enthalten.")
    if not assignments["Topic"].dropna().isin(descriptions).all():
        raise ValueError("Mindestens eine Meldung hat eine unbekannte Topic-Nummer.")
    vocabulary = np.asarray(vocabulary)
    components = np.asarray(model.components_)
    if components.shape != (12, len(vocabulary)) or not np.isfinite(components).all():
        raise ValueError("Topic-Vektoren und Vokabular passen nicht zusammen.")

    # Die führenden Begriffe dienen als Kontrolle gegen offensichtlich vertauschte Topics.
    # Bei Änderungen an den Daten oder dem Modell müssen die Beschriftungen erneut geprüft werden.
    for topic, description in descriptions.items():
        leading = vocabulary[np.argsort(components[topic - 1])[::-1][:3]]
        if set(leading) != set(description["Erwartete Begriffe"]):
            raise ValueError(f"Topic {topic} hat andere führende Begriffe. Beschriftung prüfen.")

    counts = assignments["Topic"].value_counts().reindex(range(1, 13), fill_value=0)
    numeric = summary[list(required)].copy().set_index("Topic").sort_index()
    if not np.array_equal(numeric["Anzahl Meldungen"].to_numpy(), counts.to_numpy()):
        raise ValueError("Die Häufigkeiten weichen von den aktuellen Zuordnungen ab.")
    denominator = len(documents)
    shares = (100 * counts / denominator).round(1)
    if not (numeric["Bezugsmenge Meldungen"] == denominator).all():
        raise ValueError("Die Prozentanteile haben eine abweichende Bezugsgröße.")
    if not np.allclose(numeric["Anteil Prozent"], shares, rtol=0, atol=1e-10):
        raise ValueError("Die Prozentanteile passen nicht zu den Häufigkeiten.")

    meta = pd.DataFrame.from_dict(descriptions, orient="index").rename_axis("Topic").reset_index()
    result = numeric.reset_index().merge(
        meta.drop(columns="Erwartete Begriffe"), on="Topic", how="left", validate="one_to_one"
    )
    text_columns = ["Thema", "Einordnung", "Kurzbeschreibung", "Beispielhafte Hinweise"]
    if result[text_columns].isna().any().any():
        raise ValueError("Mindestens eine Beschriftung fehlt.")
    result = result.sort_values(["Anzahl Meldungen", "Topic"], ascending=[False, True]).reset_index(drop=True)
    result.insert(0, "Rang", np.arange(1, len(result) + 1))
    return result


_required_for_labels = (
    "SELECTED_TOPIC_COUNTS", "TOPIC_LABELS_REVIEWED", "nmf_model", "nmf_assignments",
    "df_model", "nmf_summary", "tfidf_vectorizer",
)
_missing_for_labels = [name for name in _required_for_labels if name not in globals()]
if _missing_for_labels:
    raise RuntimeError("Zuerst die Schritte bis 14 ausführen. Es fehlen: " + ", ".join(_missing_for_labels))
if SELECTED_TOPIC_COUNTS.get("NMF") != 12:
    raise ValueError("Die Beschriftungen sind für NMF mit zwölf Topics vorgesehen.")
if not TOPIC_LABELS_REVIEWED:
    raise RuntimeError("Zuerst die neuen Beschriftungen einsetzen und TOPIC_LABELS_REVIEWED aktivieren.")

decision_table = build_decision_overview(
    nmf_model, tfidf_vectorizer.get_feature_names_out(), nmf_assignments,
    df_model, nmf_summary, topic_descriptions,
)
print("Beschriftete Topics:", len(decision_table))
print("Davon als gemischt gekennzeichnet:", int(decision_table["Einordnung"].eq("Gemischtes Topic").sum()))



In [ ]:
# Schritt 16 Tabelle und Übersicht für Entscheidungstragende trennen

import numpy as np
import pandas as pd


def prepare_audience_overview(table):
    """Benannte Schwerpunkte anzeigen und die übrigen Meldungen getrennt ausweisen."""
    columns = [
        "Rang", "Thema", "Anzahl Meldungen", "Anteil Prozent", "Einordnung",
        "Kurzbeschreibung", "Beispielhafte Hinweise", "Topic", "Bezugsmenge Meldungen",
    ]
    if not isinstance(table, pd.DataFrame) or not set(columns).issubset(table.columns):
        raise ValueError("Die beschriftete Tabelle aus Schritt 15 fehlt oder ist unvollständig.")
    if table.empty:
        raise ValueError("Die beschriftete Tabelle ist leer.")
    full = table[columns].copy()
    if full[columns].isna().any().any():
        raise ValueError("Die beschriftete Tabelle enthält fehlende Angaben.")
    if full["Topic"].duplicated().any():
        raise ValueError("Eine Topic-Nummer kommt mehrfach vor.")
    allowed = {"Thematischer Schwerpunkt", "Gemischtes Topic"}
    if not full["Einordnung"].isin(allowed).all():
        raise ValueError("Die Tabelle enthält eine unbekannte Einordnung.")

    counts = pd.to_numeric(full["Anzahl Meldungen"], errors="raise").to_numpy(dtype=float)
    bases = pd.to_numeric(full["Bezugsmenge Meldungen"], errors="raise").to_numpy(dtype=float)
    shares = pd.to_numeric(full["Anteil Prozent"], errors="raise").to_numpy(dtype=float)
    if not np.isfinite(counts).all() or (counts < 0).any() or (counts != np.floor(counts)).any():
        raise ValueError("Die Häufigkeiten müssen nicht negative ganze Zahlen sein.")
    if not np.isfinite(bases).all() or (bases <= 0).any() or (bases != np.floor(bases)).any():
        raise ValueError("Die Bezugsgröße muss eine positive ganze Zahl sein.")
    if not (bases == bases[0]).all():
        raise ValueError("Die Prozentwerte beziehen sich auf unterschiedliche Gesamtzahlen.")
    total = int(bases[0])
    assigned = int(counts.sum())
    if assigned > total:
        raise ValueError("Die Summe der Zuordnungen ist größer als die Bezugsgröße.")
    if not np.allclose(shares, np.round(100 * counts / total, 1), rtol=0, atol=1e-10):
        raise ValueError("Die Prozentwerte passen nicht zu den Häufigkeiten.")

    # Alle zwölf Topics bleiben in der vollständigen Tabelle erhalten.
    full = full.sort_values(["Anzahl Meldungen", "Topic"], ascending=[False, True]).reset_index(drop=True)
    full["Rang"] = np.arange(1, len(full) + 1)
    shown = full.loc[full["Einordnung"].eq("Thematischer Schwerpunkt")].copy()
    if shown.empty:
        raise ValueError("Es sind keine benannten Schwerpunkte für die Übersicht vorhanden.")
    shown = shown.reset_index(drop=True)
    shown["Rang"] = np.arange(1, len(shown) + 1)
    n_shown = int(shown["Anzahl Meldungen"].sum())
    n_mixed = int(full.loc[full["Einordnung"].eq("Gemischtes Topic"), "Anzahl Meldungen"].sum())
    n_unassigned = total - assigned
    if n_shown + n_mixed + n_unassigned != total:
        raise RuntimeError("Die Teilmengen ergeben nicht die Gesamtzahl der Meldungen.")

    # Diese Zusammenfassung beschreibt die Darstellung und nicht die Genauigkeit der Erkennung.
    coverage = pd.DataFrame({
        "Darstellung": ["In der Themenübersicht", "Nicht nach Themen aufgeschlüsselt"],
        "Anzahl Meldungen": [n_shown, n_mixed + n_unassigned],
        "Anteil Prozent": [round(100 * n_shown / total, 1), round(100 * (n_mixed + n_unassigned) / total, 1)],
        "Bezugsmenge Meldungen": [total, total],
    })
    public_columns = [
        "Rang", "Thema", "Anzahl Meldungen", "Anteil Prozent",
        "Kurzbeschreibung", "Beispielhafte Hinweise", "Bezugsmenge Meldungen",
    ]
    return full, shown[public_columns].copy(), coverage


if "decision_table" not in globals() or decision_table is None:
    raise RuntimeError("Zuerst Schritt 15 mit den aktuellen Beschriftungen ausführen.")

decision_table_all, decision_table_final, decision_coverage = prepare_audience_overview(decision_table)

decision_display = decision_table_final[["Rang", "Thema", "Anzahl Meldungen", "Anteil Prozent"]].copy()
decision_display["Anteil Prozent"] = decision_display["Anteil Prozent"].map(
    lambda value: f"{value:.1f} %".replace(".", ",")
)
display(decision_display.rename(columns={"Anteil Prozent": "Anteil"}))
print("Meldungen in dargestellten Schwerpunkten:", int(decision_coverage.iloc[0]["Anzahl Meldungen"]))
print("Nicht nach Themen aufgeschlüsselt:", int(decision_coverage.iloc[1]["Anzahl Meldungen"]))
print("Bezugsgröße der Prozentanteile:", int(decision_coverage.iloc[0]["Bezugsmenge Meldungen"]))
print("Technische Kontrolle der Darstellung: bestanden")


In [ ]:
# Schritt 17 Die Tabelle und die kommunale Übersicht speichern

from pathlib import Path

required_tables = ("decision_table_all", "decision_table_final", "decision_coverage", "OUTPUT_DIR")
missing_tables = [name for name in required_tables if name not in globals()]
if missing_tables:
    raise RuntimeError("Zuerst Schritt 16 ausführen. Es fehlen: " + ", ".join(missing_tables))

_decision_output = Path(OUTPUT_DIR)
_decision_output.mkdir(parents=True, exist_ok=True)

# Die vollständige Tabelle enthält auch die gemischten Topics und ihre Kennungen.
_export_tables = {
    "nmf_topics_beschriftet.csv": decision_table_all,
    "entscheidungstraeger_themenuebersicht.csv": decision_table_final,
    "umfang_themenuebersicht.csv": decision_coverage,
}
for _filename, _table in _export_tables.items():
    _path = _decision_output / _filename
    _table.to_csv(_path, index=False, sep=";", encoding="utf-8-sig")
    print("Gespeichert:", _path.resolve())


In [ ]:
# Schritt 18 Die Schwerpunkte als Balkendiagramm darstellen

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

_required_chart = ("decision_table_final", "decision_coverage", "OUTPUT_DIR")
_missing_chart = [name for name in _required_chart if name not in globals()]
if _missing_chart:
    raise RuntimeError("Zuerst Schritt 16 ausführen. Es fehlen: " + ", ".join(_missing_chart))

chart_df = decision_table_final.sort_values("Rang").copy()
if chart_df.empty or len(decision_coverage) != 2:
    raise ValueError("Für das Diagramm fehlen die Schwerpunkte oder die Gesamtzahlen.")
denominator = int(decision_coverage.iloc[0]["Bezugsmenge Meldungen"])
shown_count = int(chart_df["Anzahl Meldungen"].sum())
other_count = int(decision_coverage.iloc[1]["Anzahl Meldungen"])
if shown_count != int(decision_coverage.iloc[0]["Anzahl Meldungen"]) or shown_count + other_count != denominator:
    raise ValueError("Diagramm und Zusammenfassung beziehen sich auf unterschiedliche Ergebnisse.")
if not np.allclose(chart_df["Anteil Prozent"], (100 * chart_df["Anzahl Meldungen"] / denominator).round(1)):
    raise ValueError("Die Prozentwerte des Diagramms haben eine falsche Bezugsgröße.")

def format_number(value):
    return f"{int(value):,}".replace(",", ".")

def format_share(value):
    return f"{value:.1f} %".replace(".", ",")

fig, ax = plt.subplots(figsize=(12, 6.8))
positions = np.arange(len(chart_df))
bars = ax.barh(positions, chart_df["Anzahl Meldungen"])
ax.set_yticks(positions, labels=chart_df["Thema"])
ax.invert_yaxis()
ax.set_xlabel("Anzahl der automatisch zugeordneten Meldungen")
ax.set_title("Schwerpunkte der Meldungen", loc="left", pad=30, fontsize=15)
ax.text(0, 1.035, "Automatische Themenanalyse · Mängelmelder Konstanz", transform=ax.transAxes, fontsize=10)
ax.set_xlim(0, max(1, float(chart_df["Anzahl Meldungen"].max())) * 1.25)
ax.spines[["top", "right"]].set_visible(False)
ax.bar_label(
    bars,
    labels=[f"{format_number(count)} ({format_share(share)})" for count, share in zip(chart_df["Anzahl Meldungen"], chart_df["Anteil Prozent"])],
    padding=5, fontsize=10,
)

# Die nicht dargestellten Meldungen bleiben Teil der Bezugsgröße.
footer = (
    f"Dargestellt: {format_number(shown_count)} Meldungen ({format_share(100 * shown_count / denominator)}).\n"
    f"Nicht nach Themen aufgeschlüsselt: {format_number(other_count)} Meldungen ({format_share(100 * other_count / denominator)}).\n"
    f"Basis: {format_number(denominator)} Meldungen nach Bereinigung. Keine Rangfolge nach Dringlichkeit."
)
fig.text(0.035, 0.025, footer, fontsize=9.5, va="bottom", linespacing=1.6)
fig.subplots_adjust(left=0.30, right=0.98, top=0.83, bottom=0.24)
chart_path = Path(OUTPUT_DIR) / "themenhaeufigkeiten_nmf.png"
chart_path.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(chart_path, dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)
print("Diagramm gespeichert:", chart_path.resolve())